# A4/LEARN OMOP CDM Analytical Demonstration

Validates the A4/LEARN OMOP ETL by re-deriving published results from the OMOP tables, then runs one novel cross-domain analysis.

Source manuscripts (PDFs in `papers/`):
- **Sperling et al., NEJM 2023;389:1096-107** (`NEJMoa2305032.pdf`) — A4 primary results
- **Rissman et al., J Prev Alz Dis 2024;4(11):823-830** (`main.pdf`) — plasma p-tau217
- **Rentz et al., J Prev Alz Dis 2024;4(11):814-822** (`jpad.2024.123.pdf`) — CDR progression by amyloid burden

| # | Analysis | Source | Published target |
|---|----------|--------|------------------|
| 1 | Baseline characteristics (Table 1) | Sperling, Table 1 | mITT demographics by arm |
| 2 | P-tau217 predicts amyloid PET | Rissman, Fig. 1 | AUROC 0.87 (≥20 CL) / 0.89 (≥33 CL); Spearman r = 0.73 |
| 3 | CDR progression by amyloid tertile | Rentz | 19% / 35% / 41% at week 240 (modeled) |
| 4 | PACC treatment effect | Sperling, Table 2 | −0.30 (95% CI −0.82 to 0.22), p = 0.26 |
| 6–13 | Additional endpoints | all three | per section |
| 5 | Novel: multi-modal prediction | — | cross-domain ML |

Caveats that apply throughout: the trial SAP models are approximated (site is not in OMOP; simplified longitudinal models); published Table 1 uses the mITT population while OMOP holds all randomized subjects; dates are synthetic (anchored to 2020), so age is re-derived from `year_of_birth`.

---
## Setup & Shared Utilities

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# ── Paths ──
BASE_DIR = Path('..') 
OMOP_DIR = BASE_DIR / 'OMOP_Output'
DERIVED_DIR = BASE_DIR / 'Derived Data'
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

print('Loading OMOP tables...')
person = pd.read_csv(OMOP_DIR / 'person.csv')
measurement = pd.read_csv(OMOP_DIR / 'measurement.csv', low_memory=False)
observation = pd.read_csv(OMOP_DIR / 'observation.csv', low_memory=False)
visit_occurrence = pd.read_csv(OMOP_DIR / 'visit_occurrence.csv')
drug_exposure = pd.read_csv(OMOP_DIR / 'drug_exposure.csv', low_memory=False)
observation_period = pd.read_csv(OMOP_DIR / 'observation_period.csv')

print(f'  person:             {len(person):>8,} rows  ({person.person_id.nunique():,} subjects)')
print(f'  measurement:        {len(measurement):>8,} rows')
print(f'  observation:        {len(observation):>8,} rows')
print(f'  visit_occurrence:   {len(visit_occurrence):>8,} rows')
print(f'  drug_exposure:      {len(drug_exposure):>8,} rows')
print(f'  observation_period: {len(observation_period):>8,} rows')

In [ ]:
# ── Treatment Arm Lookup ──
# Both arms appear in drug_exposure (blinded trial), so we read TX from source ADQS
adqs = pd.read_csv(DERIVED_DIR / 'ADQS.csv', low_memory=False)
tx_lookup = adqs[['BID', 'TX']].dropna(subset=['TX']).drop_duplicates()

# Map BID to person_id
person_bid = person[['person_id', 'person_source_value']].copy()
person_bid.rename(columns={'person_source_value': 'BID'}, inplace=True)
tx_lookup = tx_lookup.merge(person_bid, on='BID', how='inner')

# Keep one TX per person
tx_lookup = tx_lookup.drop_duplicates(subset='person_id').set_index('person_id')['TX']
print(f'Treatment arm lookup: {len(tx_lookup)} subjects')
print(tx_lookup.value_counts())

In [ ]:
# ── Helper Functions ──

def get_measurements(source_pattern, exact=False):
    """Filter measurement table by measurement_source_value pattern."""
    if exact:
        return measurement[measurement['measurement_source_value'] == source_pattern].copy()
    return measurement[measurement['measurement_source_value'].str.contains(source_pattern, na=False)].copy()

def get_observations(source_pattern, exact=False):
    """Filter observation table by observation_source_value pattern."""
    if exact:
        return observation[observation['observation_source_value'] == source_pattern].copy()
    return observation[observation['observation_source_value'].str.contains(source_pattern, na=False)].copy()

def get_baseline(df, person_col='person_id', date_col='measurement_date'):
    """Get earliest record per person (baseline). Use for non-trial-specific analyses."""
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    idx = df.groupby(person_col)[date_col].idxmin()
    return df.loc[idx]

# ── Visit 006 = Randomization/Baseline ──
# Published A4 analyses define baseline as visit 006 (randomization), not earliest measurement.
# Visit 006 date = randomization date = time zero for treatment effect analyses.

_v006 = visit_occurrence[visit_occurrence['visit_source_value'].str.endswith('_006')]
V006_IDS = set(_v006['visit_occurrence_id'])
V006_DATES = _v006.set_index('person_id')['visit_start_date'].copy()
V006_DATES = pd.to_datetime(V006_DATES)
V006_PERSONS = set(_v006['person_id'])

print(f'Visit 006 (randomization): {len(V006_IDS)} visits, {len(V006_PERSONS)} persons')

def get_baseline_v006(df, person_col='person_id'):
    """Get measurement at visit 006 (randomization/baseline visit).
    
    Falls back to closest measurement within ±7 days of visit 006 date
    for measurements not directly linked to a visit.
    """
    df = df.copy()
    
    # Method 1: Direct visit linkage
    at_v006 = df[df['visit_occurrence_id'].isin(V006_IDS)]
    if len(at_v006) > 0:
        # One record per person (take first if duplicates)
        at_v006 = at_v006.drop_duplicates(subset=[person_col], keep='first')
        return at_v006
    
    # Method 2: Date-based matching (±7 days from visit 006 date)
    df['measurement_date'] = pd.to_datetime(df['measurement_date'])
    df = df[df[person_col].isin(V006_PERSONS)]
    df['rand_date'] = df[person_col].map(V006_DATES)
    df['days_from_rand'] = (df['measurement_date'] - df['rand_date']).dt.days.abs()
    df = df[df['days_from_rand'] <= 7]
    idx = df.groupby(person_col)['days_from_rand'].idxmin()
    return df.loc[idx].drop(columns=['rand_date', 'days_from_rand'])

def suvr_to_centiloids(suvr):
    """Convert florbetapir SUVR to centiloids (Navitsky et al.)."""
    return 196.325 * suvr - 196.325

def get_randomized_persons():
    """Get person_ids of randomized subjects (those with TX assignment)."""
    return set(tx_lookup.index)

def mean_sd(series):
    """Format mean (SD) string."""
    return f"{series.mean():.1f} ({series.std():.1f})"

print('Utilities loaded.')

---
## Analysis 1: Baseline Characteristics (Table 1)

**Source**: Sperling et al., NEJM 2023, Table 1 (p. 1100).

Published Table 1 describes the modified intention-to-treat population (N = 564 solanezumab / 583 placebo). OMOP holds all randomized subjects (578 / 591), so small differences are expected.

In [ ]:
# ── Identify randomized subjects ──
randomized_ids = get_randomized_persons()
person_rand = person[person['person_id'].isin(randomized_ids)].copy()
person_rand['TX'] = person_rand['person_id'].map(tx_lookup)

print(f'Randomized subjects in OMOP: {len(person_rand)}')
print(f'By arm:\n{person_rand["TX"].value_counts()}')

In [ ]:
# ── Demographics ──
# Dates are anchored to a synthetic 2020 baseline (helpers.py), so
# 2020 - year_of_birth recovers age at baseline.
person_rand['age_approx'] = 2020 - person_rand['year_of_birth']

# Sex: gender_concept_id 8532 = Female, 8507 = Male
person_rand['female'] = (person_rand['gender_concept_id'] == 8532).astype(int)

# ── Baseline cognitive scores ──
# MMSE and PACC are collected at visit 006 (randomization)
# MMSE totals moved to OBSERVATION on 2026-08-21 (42869860 is an
# Observation-domain concept); rename the date column so the shared
# baseline helpers keep working.
mmse = (get_observations('MMSE:MMSCORE', exact=True)
        .rename(columns={'observation_date': 'measurement_date'}))
mmse_bl = get_baseline_v006(mmse)
mmse_bl = mmse_bl[mmse_bl['person_id'].isin(randomized_ids)]

pacc = get_measurements('PACC:PACC.raw', exact=True)
pacc_bl = get_baseline_v006(pacc)
pacc_bl = pacc_bl[pacc_bl['person_id'].isin(randomized_ids)]

# CDR-SB is collected at screening (visit 001), not visit 006
# Use earliest (screening) CDR as baseline, matching published Table 1
cdrsb = get_measurements('CDR:CDSOB')
cdrsb_primary = cdrsb[cdrsb['measurement_source_value'].str.contains('SP=1.0.*in-person', na=False)]
if len(cdrsb_primary) > 0:
    cdrsb = cdrsb_primary
cdrsb_bl = get_baseline(cdrsb)
cdrsb_bl = cdrsb_bl[cdrsb_bl['person_id'].isin(randomized_ids)]

# ── Amyloid PET (screening, not visit 006 — PET done at screening visits) ──
amyloid = get_measurements('AMYLOID\\|Florbetapir\\|Composite_Summary')
amyloid_bl = get_baseline(amyloid)
amyloid_bl = amyloid_bl[amyloid_bl['person_id'].isin(randomized_ids)]
amyloid_bl['centiloids'] = suvr_to_centiloids(amyloid_bl['value_as_number'])

# ── APOE e4 carrier (time-invariant, earliest is fine) ──
apoe_carrier = get_measurements('ADQS:APOEGNPRSNFLG')
apoe_bl = get_baseline(apoe_carrier)
apoe_bl = apoe_bl[apoe_bl['person_id'].isin(randomized_ids)]

print(f'Baseline data available:')
print(f'  MMSE:        {len(mmse_bl)} subjects (visit 006)')
print(f'  PACC:        {len(pacc_bl)} subjects (visit 006)')
print(f'  CDR-SB:      {len(cdrsb_bl)} subjects (screening)')
print(f'  Amyloid PET: {len(amyloid_bl)} subjects (screening)')
print(f'  APOE e4:     {len(apoe_bl)} subjects')

In [ ]:
# ── Build Table 1 ──
def build_arm_stats(arm_name, person_arm, mmse_bl, pacc_bl, cdrsb_bl, amyloid_bl, apoe_bl):
    """Compute summary statistics for one treatment arm."""
    pids = set(person_arm['person_id'])
    n = len(person_arm)
    
    row = {'N': n}
    row['Age, mean (SD)'] = mean_sd(person_arm['age_approx'])
    row['Female, %'] = f"{100 * person_arm['female'].mean():.1f}%"
    
    # APOE
    apoe_arm = apoe_bl[apoe_bl['person_id'].isin(pids)]
    if len(apoe_arm) > 0:
        row['APOE e4 carrier, %'] = f"{100 * apoe_arm['value_as_number'].mean():.1f}%"
    else:
        row['APOE e4 carrier, %'] = 'N/A'
    
    # MMSE
    mmse_arm = mmse_bl[mmse_bl['person_id'].isin(pids)]
    row['MMSE, mean (SD)'] = mean_sd(mmse_arm['value_as_number']) if len(mmse_arm) > 0 else 'N/A'
    
    # PACC
    pacc_arm = pacc_bl[pacc_bl['person_id'].isin(pids)]
    row['PACC composite, mean (SD)'] = mean_sd(pacc_arm['value_as_number']) if len(pacc_arm) > 0 else 'N/A'
    
    # CDR-SB
    cdrsb_arm = cdrsb_bl[cdrsb_bl['person_id'].isin(pids)]
    row['CDR-SB, mean (SD)'] = mean_sd(cdrsb_arm['value_as_number']) if len(cdrsb_arm) > 0 else 'N/A'
    
    # Amyloid
    amy_arm = amyloid_bl[amyloid_bl['person_id'].isin(pids)]
    row['Amyloid PET (CL), mean (SD)'] = mean_sd(amy_arm['centiloids']) if len(amy_arm) > 0 else 'N/A'
    
    return row

sol_persons = person_rand[person_rand['TX'] == 'Solanezumab']
pla_persons = person_rand[person_rand['TX'] == 'Placebo']

sol_stats = build_arm_stats('Solanezumab', sol_persons, mmse_bl, pacc_bl, cdrsb_bl, amyloid_bl, apoe_bl)
pla_stats = build_arm_stats('Placebo', pla_persons, mmse_bl, pacc_bl, cdrsb_bl, amyloid_bl, apoe_bl)

# Published values from NEJM 2023 Table 1 (mITT population, N=564/583)
published = {
    'N': '564 / 583 (mITT)',
    'Age, mean (SD)': '72.0 (4.7) / 71.9 (5.0)',
    'Female, %': '58.3% / 60.4%',
    'APOE e4 carrier, %': '59.0% / 58.7%',
    'MMSE, mean (SD)': '28.8 (1.3) / 28.8 (1.2)',
    'PACC composite, mean (SD)': '0.0 (2.8) / 0.0 (2.6)',
    'CDR-SB, mean (SD)': '0.1 (0.2) / 0.0 (0.2)',
    'Amyloid PET (CL), mean (SD)': '66.2 (33.5) / 65.9 (32.1)',
}

table1 = pd.DataFrame({
    'Solanezumab (OMOP)': sol_stats,
    'Placebo (OMOP)': pla_stats,
    'Published mITT (Sol / Pla)': published,
})

print('\n=== TABLE 1: Baseline Characteristics by Treatment Arm ===')
print('\nComparing OMOP CDM values vs. NEJM 2023 published values:\n')
display(table1)

---
## Analysis 2: P-tau217 Prediction of Amyloid PET Status

**Source**: Rissman et al., JPAD 2024 — baseline plasma p-tau217 AUROC 0.87 (95% CI 0.85–0.88) for amyloid PET ≥20 CL and 0.89 (0.87–0.90) for ≥33 CL; Spearman r = 0.73 (0.71–0.75) against centiloids.

The published analysis pooled baseline samples from A4 randomized subjects (both arms) with the amyloid-negative LEARN cohort (total N = 1626); LEARN supplies most of the negative class. Merging baseline p-tau217 with amyloid PET below reproduces that A4+LEARN mix.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# ── Extract P-tau217 ──
ptau217 = get_measurements('PTAU217')
print(f'P-tau217 records: {len(ptau217)} ({ptau217.person_id.nunique()} persons)')
print(f'Source values: {ptau217.measurement_source_value.unique()[:3]}')

ptau217_bl = get_baseline(ptau217)
ptau217_bl = ptau217_bl[['person_id', 'value_as_number']].rename(
    columns={'value_as_number': 'ptau217'}
)

# ── Extract Amyloid PET composite ──
amyloid_all = get_measurements('AMYLOID\\|Florbetapir\\|Composite_Summary')
print(f'Amyloid PET records: {len(amyloid_all)} ({amyloid_all.person_id.nunique()} persons)')

amyloid_pet_bl = get_baseline(amyloid_all)
amyloid_pet_bl = amyloid_pet_bl[['person_id', 'value_as_number']].copy()
amyloid_pet_bl['centiloids'] = suvr_to_centiloids(amyloid_pet_bl['value_as_number'])
amyloid_pet_bl.rename(columns={'value_as_number': 'amyloid_suvr'}, inplace=True)

# ── Merge ──
roc_df = ptau217_bl.merge(amyloid_pet_bl[['person_id', 'centiloids']], on='person_id', how='inner')
roc_df = roc_df.dropna(subset=['ptau217', 'centiloids'])
print(f'\nMatched subjects with both P-tau217 and amyloid PET: {len(roc_df)}')

In [ ]:
# ── ROC Analysis ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, threshold, published_auc in [(axes[0], 20, 0.87), (axes[1], 33, 0.89)]:
    y_true = (roc_df['centiloids'] >= threshold).astype(int)
    y_score = roc_df['ptau217']
    
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    
    ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'OMOP AUROC = {auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.axhline(y=published_auc, color='r', linestyle=':', alpha=0.5, label=f'Published AUROC = {published_auc:.2f}')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'P-tau217 → Amyloid PET ≥{threshold} CL\n(n={len(roc_df)}, {y_true.sum()} positive)')
    ax.legend(loc='lower right')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis2_ptau217_roc.png', bbox_inches='tight')
plt.show()

# ── Correlation ──
# Published (Rissman et al., JPAD 2024) used Spearman's rank correlation, not Pearson.
# P-tau217 is right-skewed, making Spearman more appropriate and yielding higher r.
r_spearman, p_spearman = stats.spearmanr(roc_df['ptau217'], roc_df['centiloids'])
r_pearson, p_pearson = stats.pearsonr(roc_df['ptau217'], roc_df['centiloids'])
r, p = r_spearman, p_spearman  # use Spearman as primary (matches published method)

print(f'\nSpearman correlation (P-tau217 vs centiloids): r = {r_spearman:.3f}, p = {p_spearman:.2e}')
print(f'Pearson correlation  (P-tau217 vs centiloids): r = {r_pearson:.3f}, p = {p_pearson:.2e}')
print(f'Published (Spearman): r = 0.73 (95% CI: 0.71 to 0.75)')

In [ ]:
# ── Scatter plot ──
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(roc_df['ptau217'], roc_df['centiloids'], alpha=0.3, s=10)
ax.axhline(y=20, color='orange', linestyle='--', label='20 CL threshold')
ax.axhline(y=33, color='red', linestyle='--', label='33 CL threshold')

# Fit line
z = np.polyfit(roc_df['ptau217'], roc_df['centiloids'], 1)
x_line = np.linspace(roc_df['ptau217'].min(), roc_df['ptau217'].max(), 100)
ax.plot(x_line, np.polyval(z, x_line), 'r-', linewidth=2, alpha=0.7)

ax.set_xlabel('Plasma P-tau217 (pg/mL)')
ax.set_ylabel('Amyloid PET (Centiloids)')
ax.set_title(f'P-tau217 vs Amyloid PET (Spearman r={r_spearman:.3f}, Pearson r={r_pearson:.3f}, n={len(roc_df)})')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis2_ptau217_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── CSF Aβ42/40 vs amyloid PET (Rissman et al., Fig. 1B) ──
# Published: Spearman r = -0.54 (95% CI -0.62 to -0.45) in the A4 CSF substudy.
csf42_bl = get_baseline(get_measurements('SRT15754: CSF Mod. A-Beta1-42-221', exact=True))
csf40_bl = get_baseline(get_measurements('SRT15753: CSF Mod. A-Beta1-40-221', exact=True))
csf_ratio = (csf42_bl.set_index('person_id')['value_as_number']
             / csf40_bl.set_index('person_id')['value_as_number']).rename('csf_ab4240')

csf_df = pd.DataFrame({'csf_ab4240': csf_ratio}).join(
    amyloid_pet_bl.set_index('person_id')['centiloids'], how='inner').dropna()
r_csf, p_csf = stats.spearmanr(csf_df['csf_ab4240'], csf_df['centiloids'])
print(f'CSF Aβ42/40 vs amyloid PET: Spearman r = {r_csf:.2f} (n={len(csf_df)}, p={p_csf:.1e})')
print(f'Published (Rissman Fig. 1B, CSF substudy): r = -0.54 (-0.62 to -0.45)')

---
## Analysis 3: CDR Progression by Amyloid Burden Tertile

**Source**: Rentz et al., JPAD 2024 — GEE-modeled probability of CDR-Global progression at week 240 by baseline amyloid tertile (cutpoints 46.1 and 77.2 CL): 19% / 35% / 41%.

Definitions from the paper:
- Progressor = non-zero CDR-G at **two consecutive visits**
- Both treatment arms combined (the paper found no arm differences in CDR outcomes)
- Tertile estimates come from refitting the model within each tertile

We check the published modeled rates two ways: nonparametric Kaplan-Meier incidence, then a GEE refit per tertile.

In [ ]:
# ── Baseline amyloid PET → tertiles ──
# Rentz et al. split the randomized population (both arms) at 46.1 and 77.2 CL.
# A4 amyloid eligibility was SUVR ≥1.15 (or 1.10-1.15 plus positive visual read);
# the ≥20 CL filter below approximates that eligibility on the centiloid scale.
amy_bl = get_baseline(get_measurements('AMYLOID\\|Florbetapir\\|Composite_Summary'))
amy_bl = amy_bl[['person_id', 'value_as_number']].copy()
amy_bl['centiloids'] = suvr_to_centiloids(amy_bl['value_as_number'])
amy_bl = amy_bl.dropna(subset=['centiloids'])

print(f'All subjects with amyloid PET: {len(amy_bl)}')
print(f'Centiloid range: {amy_bl["centiloids"].min():.1f} to {amy_bl["centiloids"].max():.1f}')

# Filter to amyloid-positive (≥20 CL) to match A4 trial population
amy_bl_pos = amy_bl[amy_bl['centiloids'] >= 20].copy()
print(f'Amyloid-positive (≥20 CL): {len(amy_bl_pos)}')

# Use PUBLISHED cutpoints (46.1 and 77.2 CL) for direct replication
PUB_CUT_LOW = 46.1
PUB_CUT_HIGH = 77.2

amy_bl_pos['tertile'] = pd.cut(
    amy_bl_pos['centiloids'],
    bins=[0, PUB_CUT_LOW, PUB_CUT_HIGH, 999],
    labels=['Low', 'Intermediate', 'High'],
    right=False
)

tertile_cuts = amy_bl_pos.groupby('tertile')['centiloids'].agg(['min', 'max', 'count'])

print(f'\nAmyloid tertile cutpoints (using published cutpoints):')
print(f'  Low:          20.0 - {PUB_CUT_LOW} CL')
print(f'  Intermediate: {PUB_CUT_LOW} - {PUB_CUT_HIGH} CL')
print(f'  High:         ≥{PUB_CUT_HIGH} CL')
display(tertile_cuts)

# Also show what data-driven tertiles would give
data_driven = pd.qcut(amy_bl_pos['centiloids'], q=3, retbins=True)[1]
print(f'\nData-driven tertile cuts: {data_driven[1]:.1f}, {data_driven[2]:.1f} CL')
print(f'Published cuts:          {PUB_CUT_LOW}, {PUB_CUT_HIGH} CL')

In [ ]:
# ── Longitudinal CDR Global ──
# CDR is collected at screening (visit 001) and then at visits 018, 033, 048, 057, 066...
# NOT at visit 006 (randomization), so baseline CDR = screening CDR (visit 001).
# We keep primary study partner (SP=1.0) records, in-person and telephone; Rentz et al.
# do not specify rater filtering, so this follows the source data dictionary.
cdr_global = get_measurements('CDR:CDGLOBAL')

# Filter to primary study partner (SP=1.0) — both in-person and telephone
# (both administration modes kept)
cdr_primary = cdr_global[cdr_global['measurement_source_value'].str.contains('SP=1.0', na=False)]
if len(cdr_primary) > 0:
    cdr_global = cdr_primary

print(f'CDR Global records (SP=1.0, all modes): {len(cdr_global)} ({cdr_global.person_id.nunique()} persons)')
if len(cdr_global) > 0:
    print(f'Value distribution:\n{cdr_global.value_as_number.describe()}')
    print(f'\nAssessment modes:')
    for sv in sorted(cdr_global['measurement_source_value'].unique()):
        n = len(cdr_global[cdr_global['measurement_source_value'] == sv])
        print(f'  {sv}: n={n}')

# ── Baseline CDR at screening (earliest, visit 001) ──
cdr_bl = get_baseline(cdr_global)
print(f'\nCDR at screening (earliest): {len(cdr_bl)} persons')
print(f'CDR=0 at baseline: {(cdr_bl["value_as_number"] == 0).sum()}')

In [ ]:
# ── CDR progression analysis with Kaplan-Meier (censoring-aware) ──
# Manual KM implementation (lifelines requires Python 3.10+)
# Published analysis (Rentz et al., JPAD 2024) used BOTH treatment arms combined,
# restricted to RANDOMIZED subjects with CDR-G=0 at baseline.
# Progression = CDR-G > 0 at TWO CONSECUTIVE visits (not single visit).

def kaplan_meier(time, event):
    """Compute Kaplan-Meier survival function with Greenwood CI."""
    df = pd.DataFrame({'time': time, 'event': event}).sort_values('time')
    times = sorted(df['time'].unique())
    n = len(df)
    
    records = [{'time': 0, 'survival': 1.0, 'var_sum': 0.0, 'n_at_risk': n, 'n_events': 0}]
    n_at_risk = n
    surv = 1.0
    var_sum = 0.0
    
    for t in times:
        at_t = df[df['time'] == t]
        d = at_t['event'].sum()
        c = len(at_t) - d
        if n_at_risk > 0 and d > 0:
            surv *= (1 - d / n_at_risk)
            if n_at_risk > d:
                var_sum += d / (n_at_risk * (n_at_risk - d))
        records.append({'time': t, 'survival': surv, 'var_sum': var_sum,
                        'n_at_risk': n_at_risk, 'n_events': d})
        n_at_risk -= (d + c)
    
    result = pd.DataFrame(records)
    result['se'] = result['survival'] * np.sqrt(result['var_sum'])
    result['ci_lower'] = np.clip(result['survival'] - 1.96 * result['se'], 0, 1)
    result['ci_upper'] = np.clip(result['survival'] + 1.96 * result['se'], 0, 1)
    return result

def km_survival_at(km_df, target_time):
    """Get survival estimate at a specific time (step function)."""
    km_before = km_df[km_df['time'] <= target_time]
    if len(km_before) == 0:
        return 1.0
    return km_before.iloc[-1]['survival']


if len(cdr_global) > 0:
    cdr_global = cdr_global.copy()
    cdr_global['measurement_date'] = pd.to_datetime(cdr_global['measurement_date'])
    
    cdr_bl_0 = cdr_bl[cdr_bl['value_as_number'] == 0]
    persons_cdr0 = set(cdr_bl_0['person_id'])
    
    cdr_fu = cdr_global[cdr_global['person_id'].isin(persons_cdr0)].copy()
    bl_dates_cdr = cdr_bl_0.set_index('person_id')['measurement_date']
    bl_dates_cdr = pd.to_datetime(bl_dates_cdr)
    cdr_fu['bl_date'] = cdr_fu['person_id'].map(bl_dates_cdr)
    cdr_fu = cdr_fu[cdr_fu['measurement_date'] > cdr_fu['bl_date']]
    
    # RANDOMIZED subjects only, both arms combined, amyloid-positive, CDR=0 at baseline
    persons_with_both = (persons_cdr0 
                         & set(amy_bl_pos['person_id']) 
                         & set(cdr_fu['person_id'].unique())
                         & randomized_ids)
    
    print(f'CDR analysis population: {len(persons_with_both)} randomized, amyloid+, CDR=0 at baseline')
    
    cdr_fu_sorted = cdr_fu.sort_values(['person_id', 'measurement_date'])
    
    # ── CONFIRMED progression: TWO CONSECUTIVE visits with CDR-G > 0 ──
    # Published (Rentz et al.): "a non-zero score in CDR-G at two consecutive visits"
    # This is stricter than single-visit CDR > 0 — filters transient fluctuations
    survival_records = []
    for pid in persons_with_both:
        bl_date = bl_dates_cdr.get(pid)
        if pd.isna(bl_date):
            continue
        
        pid_cdr = cdr_fu_sorted[cdr_fu_sorted['person_id'] == pid].copy()
        if len(pid_cdr) == 0:
            continue
        
        # Find first CONFIRMED CDR > 0: two consecutive visits with CDR > 0
        confirmed_date = None
        pid_dates = pid_cdr['measurement_date'].values
        pid_vals = pid_cdr['value_as_number'].values
        
        for i in range(len(pid_vals) - 1):
            if pid_vals[i] > 0 and pid_vals[i + 1] > 0:
                confirmed_date = pid_dates[i]
                break
        
        if confirmed_date is not None:
            event_date = confirmed_date
            event = 1
        else:
            event_date = pid_cdr['measurement_date'].max()
            event = 0
        
        time_weeks = (pd.Timestamp(event_date) - pd.Timestamp(bl_date)).days / 7
        if time_weeks <= 0:
            continue
        
        survival_records.append({
            'person_id': pid, 'time_weeks': time_weeks, 'event': event,
        })
    
    survival_df = pd.DataFrame(survival_records)
    survival_df = survival_df.merge(
        amy_bl_pos[['person_id', 'tertile', 'centiloids']],
        on='person_id', how='inner'
    )
    
    # ── KM estimates at 240 weeks per tertile ──
    print('=== CDR-Global Progression by Amyloid Tertile (Kaplan-Meier) ===')
    print(f'Population: RANDOMIZED, amyloid-positive (≥20 CL), BOTH arms, CDR=0 at baseline')
    print(f'Progression: TWO CONSECUTIVE visits with CDR-G > 0 (Rentz et al. definition)')
    print(f'Subjects: {len(survival_df)} (events: {survival_df["event"].sum()}, censored: {(1-survival_df["event"]).sum()})')
    
    km_results = {}
    km_curves = {}
    target_week = 240
    
    for tertile_name in ['Low', 'Intermediate', 'High']:
        tdf = survival_df[survival_df['tertile'] == tertile_name]
        km_df = kaplan_meier(tdf['time_weeks'].values, tdf['event'].values)
        km_curves[tertile_name] = km_df
        
        surv_at_target = km_survival_at(km_df, target_week)
        cum_incidence = (1 - surv_at_target) * 100
        
        km_results[tertile_name] = {
            'KM_rate_%': round(cum_incidence, 1),
            'N_total': len(tdf),
            'N_events': int(tdf['event'].sum()),
            'N_censored': int((1 - tdf['event']).sum()),
            'median_followup_wk': round(tdf['time_weeks'].median(), 0),
        }
    
    km_rates = pd.DataFrame(km_results).T
    km_rates.index.name = 'tertile'
    
    print(f'\nKaplan-Meier cumulative incidence at {target_week} weeks:\n')
    display(km_rates)
    
    published_rates = {'Low': 19, 'Intermediate': 35, 'High': 41}
    print(f'\nComparison to published (Rentz et al. 2024 — published rates are GEE-modeled):')
    for tertile_name in ['Low', 'Intermediate', 'High']:
        omop_rate = km_results[tertile_name]['KM_rate_%']
        pub_rate = published_rates[tertile_name]
        diff_pp = omop_rate - pub_rate
        status = 'PASS' if abs(diff_pp) <= 5 else 'CHECK'
        print(f'  {tertile_name:15s}: OMOP KM={omop_rate:.1f}%, Published={pub_rate}%, diff={diff_pp:+.1f}pp [{status}]')

else:
    print('CDR Global data not found.')

In [ ]:
# ── Kaplan-Meier progression curves ──
if len(cdr_global) > 0 and len(survival_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = {'Low': '#2ecc71', 'Intermediate': '#f39c12', 'High': '#e74c3c'}
    
    for tertile_name in ['Low', 'Intermediate', 'High']:
        km_df = km_curves[tertile_name]
        km_rate = km_results[tertile_name]['KM_rate_%']
        
        # Plot cumulative incidence (1 - survival) as step function
        ci = (1 - km_df['survival']) * 100
        ax.step(km_df['time'], ci, where='post',
                color=colors[tertile_name], linewidth=2,
                label=f'{tertile_name} ({km_rate:.0f}% at 240wk)')
        
        # Confidence interval (1 - upper_surv to 1 - lower_surv)
        ci_lower = (1 - km_df['ci_upper']) * 100
        ci_upper = (1 - km_df['ci_lower']) * 100
        ax.fill_between(km_df['time'], ci_lower, ci_upper,
                        color=colors[tertile_name], alpha=0.1, step='post')
    
    ax.set_xlabel('Weeks from Screening')
    ax.set_ylabel('Cumulative CDR Progression (%)')
    ax.set_title('CDR-Global Confirmed Progression by Baseline Amyloid Tertile\n(KM, randomized, both arms, amyloid+, CDR=0 at baseline, 2-visit confirmed)')
    ax.legend(title='Amyloid Tertile', loc='upper left')
    ax.set_xlim(0, 320)
    ax.set_ylim(0, 70)
    ax.axvline(x=240, color='gray', linestyle=':', alpha=0.5)
    
    # Reference lines for published rates at 240 weeks
    ax.axhline(y=19, color='#2ecc71', linestyle=':', alpha=0.3)
    ax.axhline(y=35, color='#f39c12', linestyle=':', alpha=0.3)
    ax.axhline(y=41, color='#e74c3c', linestyle=':', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'analysis3_cdr_progression_km.png', bbox_inches='tight')
    plt.show()
else:
    print('Skipping KM plot (insufficient CDR data).')

In [ ]:
# ── GEE: CDR progression (Rentz et al.) ──
# Paper methods: logistic GEE for binary CDR outcomes with unstructured working
# correlation; natural cubic splines for time (2-3 df chosen by AIC); covariates
# age, education, APOE e4, treatment, and time x treatment; tertile estimates from
# REFITTING the model within each baseline amyloid tertile.
# Deviations here: Independence working correlation with robust sandwich SEs
# (point estimates remain consistent; statsmodels has no practical unstructured
# option for irregular visit times) and fixed 2-df splines.

from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.cov_struct import Independence
from statsmodels.genmod.families import Binomial
from scipy.special import expit
import statsmodels.api as sm

def rcs_basis(x, knots):
    """Restricted (natural) cubic spline basis: k knots -> k-1 columns.
    Harrell, Regression Modeling Strategies, 2nd ed, Sec 2.4.5."""
    x = np.asarray(x, dtype=float)
    def _tc(u): return np.where(u > 0, u**3, 0.0)
    t_last, t_2nd = knots[-1], knots[-2]
    denom = t_last - t_2nd
    basis = [x.copy()]
    for j in range(len(knots) - 2):
        h = (_tc(x - knots[j])
             - ((t_last - knots[j]) / denom) * _tc(x - t_2nd)
             + ((t_2nd - knots[j]) / denom) * _tc(x - t_last))
        basis.append(h)
    return np.column_stack(basis)

# ── Education (years, EDCCNTU) from source ──
_subj = pd.read_csv(DERIVED_DIR / 'SUBJINFO.csv', low_memory=False)
_pbid = person[['person_id', 'person_source_value']].rename(columns={'person_source_value': 'BID'})
edu_map = _subj[['BID', 'EDCCNTU']].drop_duplicates().merge(_pbid, on='BID').set_index('person_id')['EDCCNTU']

# ── Person-visit dataset: has confirmed progression occurred by this visit? ──
gee_rows = []
for pid in persons_with_both:
    bl_d = bl_dates_cdr.get(pid)
    if pd.isna(bl_d):
        continue
    pid_cdr = cdr_fu_sorted[cdr_fu_sorted['person_id'] == pid]
    dates = pid_cdr['measurement_date'].values
    vals = pid_cdr['value_as_number'].values
    confirmed_yet = False
    for i in range(len(vals)):
        tw = (pd.Timestamp(dates[i]) - pd.Timestamp(bl_d)).days / 7
        if tw <= 0:
            continue
        if not confirmed_yet and i < len(vals) - 1 and vals[i] > 0 and vals[i + 1] > 0:
            confirmed_yet = True
        gee_rows.append({'person_id': pid, 'time_weeks': tw, 'cdr_pos': int(confirmed_yet)})

gee_df = pd.DataFrame(gee_rows).merge(
    amy_bl_pos[['person_id', 'tertile', 'centiloids']], on='person_id')
gee_df['age'] = 2020 - gee_df['person_id'].map(person.set_index('person_id')['year_of_birth'])
gee_df['education'] = gee_df['person_id'].map(edu_map)
gee_df['apoe_e4'] = gee_df['person_id'].map(apoe_bl.set_index('person_id')['value_as_number'])
gee_df['tx_code'] = gee_df['person_id'].map(tx_lookup).map({'Solanezumab': 1, 'Placebo': 0})
gee_df = gee_df.dropna().sort_values(['person_id', 'time_weeks']).reset_index(drop=True)
print(f'GEE dataset: {len(gee_df)} obs, {gee_df.person_id.nunique()} randomized persons; '
      f'confirmed progression {gee_df.groupby("person_id")["cdr_pos"].max().mean():.1%}')

# ── Spline basis (2 df; knots at 10/50/90th pctile of visit times) ──
time_knots = np.percentile(gee_df['time_weeks'], [10, 50, 90])
_sp = rcs_basis(gee_df['time_weeks'].values, time_knots)
gee_df['sp1'], gee_df['sp2'] = _sp[:, 0], _sp[:, 1]
_sp240 = rcs_basis(np.array([240.0]), time_knots)
sp1_240, sp2_240 = _sp240[0, 0], _sp240[0, 1]

# ── Refit within each tertile (per the paper), predict progression at week 240 ──
published_gee = {'Low': 19, 'Intermediate': 35, 'High': 41}
gee_pred_rates = {}
print('\n--- GEE-modeled confirmed CDR progression at 240 weeks (refit per tertile) ---')
for tname in ['Low', 'Intermediate', 'High']:
    tdf = gee_df[gee_df['tertile'] == tname].copy()
    tdf['age_c'] = tdf['age'] - tdf['age'].mean()
    tdf['edu_c'] = tdf['education'] - tdf['education'].mean()
    tdf['sp1_tx'] = tdf['sp1'] * tdf['tx_code']
    tdf['sp2_tx'] = tdf['sp2'] * tdf['tx_code']
    X = sm.add_constant(tdf[['sp1', 'sp2', 'age_c', 'edu_c', 'apoe_e4',
                             'tx_code', 'sp1_tx', 'sp2_tx']])
    res = GEE(tdf['cdr_pos'].values, X.values, groups=tdf['person_id'].values,
              family=Binomial(), cov_struct=Independence()).fit(maxiter=200)
    # predict at week 240: covariates at tertile means, arms averaged
    apoe_mean = tdf.groupby('person_id')['apoe_e4'].first().mean()
    pred = np.array([1, sp1_240, sp2_240, 0, 0, apoe_mean,
                     0.5, sp1_240 * 0.5, sp2_240 * 0.5])
    prob = expit(pred @ res.params) * 100
    gee_pred_rates[tname] = prob
    print(f'  {tname:<13} GEE={prob:.1f}%  published={published_gee[tname]}%  '
          f'({prob - published_gee[tname]:+.1f}pp)')

print('\n--- Method comparison ---')
print(f'{"Tertile":<15}{"KM":<10}{"GEE":<10}{"Published":<10}')
for t in ['Low', 'Intermediate', 'High']:
    print(f'{t:<15}{km_results[t]["KM_rate_%"]:<10.1f}{gee_pred_rates[t]:<10.1f}{published_gee[t]:<10}')

---
## Analysis 4: Primary Endpoint — PACC by Treatment Arm

**Source**: Sperling et al., NEJM 2023 — PACC change at 240 weeks: solanezumab −1.43, placebo −1.13, difference −0.30 (95% CI −0.82 to 0.22), p = 0.26.

The published primary analysis is a constrained longitudinal data analysis (cLDA) with natural cubic splines of days since baseline (SAP-amended; exact covariates live in the SAP, not the paper; randomization was stratified by APOE e4, education >12 y, and site). We approximate it with a change-from-baseline mixed model with spline time and the stratification factors; site is not in OMOP.

In [ ]:
# ── Longitudinal PACC by treatment arm ──
pacc_all = get_measurements('PACC:PACC.raw', exact=True).copy()
pacc_all['measurement_date'] = pd.to_datetime(pacc_all['measurement_date'])
pacc_rand = pacc_all[pacc_all['person_id'].isin(randomized_ids)].copy()
pacc_rand['TX'] = pacc_rand['person_id'].map(tx_lookup)

print(f'Longitudinal PACC records (randomized): {len(pacc_rand)} ({pacc_rand.person_id.nunique()} subjects)')

# ── Baseline PACC at visit 006 (randomization) ──
pacc_bl_v006 = get_baseline_v006(pacc_all)
pacc_bl_v006 = pacc_bl_v006[pacc_bl_v006['person_id'].isin(randomized_ids)]
pacc_bl_vals = pacc_bl_v006.set_index('person_id')['value_as_number'].rename('pacc_baseline')

pacc_rand = pacc_rand.merge(pacc_bl_vals, on='person_id', how='left')
pacc_rand['pacc_change'] = pacc_rand['value_as_number'] - pacc_rand['pacc_baseline']

# ── Extract VISCODE from visit linkage ──
# visit_source_value = "BID_VVV" where VVV is the VISCODE
visit_viscode = visit_occurrence[['visit_occurrence_id', 'visit_source_value']].copy()
visit_viscode['viscode'] = visit_viscode['visit_source_value'].str.extract(r'_(\d+)$').astype(float)
viscode_map = visit_viscode.set_index('visit_occurrence_id')['viscode']

pacc_rand['viscode'] = pacc_rand['visit_occurrence_id'].map(viscode_map)

# ── Map VISCODE to approximate weeks from randomization (visit 006) ──
# VISCODE 006 = week 0. A4 visits are roughly every 12-24 weeks.
# Key viscodes: 006=0w, 018=24w, 021=36w, 024=48w, 027=60w, 030=72w, 033=84w, ...
# Pattern: (viscode - 6) * 4 ≈ weeks (each VISCODE step ≈ 4 weeks)
# More precisely, use days from randomization for the x-axis
pacc_rand['rand_date'] = pacc_rand['person_id'].map(V006_DATES)
pacc_rand['weeks'] = (pacc_rand['measurement_date'] - pacc_rand['rand_date']).dt.days / 7

# Filter: only subjects with visit 006 (randomized) and post-baseline visits
pacc_rand = pacc_rand.dropna(subset=['rand_date', 'pacc_baseline'])
pacc_rand = pacc_rand[pacc_rand['weeks'] >= -1].copy()

# ── Group by VISCODE for trajectory (avoids binning artifacts) ──
# Filter to standard protocol visits (006, 018, 021, 024, ..., 066)
pacc_rand = pacc_rand.dropna(subset=['viscode'])
pacc_rand['viscode_int'] = pacc_rand['viscode'].astype(int)

# Compute median weeks per VISCODE for the x-axis
viscode_weeks = pacc_rand.groupby('viscode_int')['weeks'].median().rename('median_weeks')
pacc_rand = pacc_rand.merge(viscode_weeks, left_on='viscode_int', right_index=True, how='left')

print(f'PACC baseline at visit 006: {len(pacc_bl_v006)} subjects')
print(f'Post-baseline records with VISCODE: {len(pacc_rand)}')
print(f'VISCODEs present: {sorted(pacc_rand["viscode_int"].unique())}')
print(f'\nVISCODE → Median weeks from randomization:')
for vc, wk in viscode_weeks.sort_index().items():
    n = len(pacc_rand[pacc_rand['viscode_int'] == vc])
    print(f'  VISCODE {vc:03d}: ~{wk:.0f} weeks (n={n})')

In [ ]:
# ── PACC trajectory plot (grouped by VISCODE, x-axis = median weeks) ──
summary = pacc_rand.groupby(['TX', 'viscode_int']).agg(
    mean_change=('pacc_change', 'mean'),
    std_change=('pacc_change', 'std'),
    count=('pacc_change', 'count'),
    median_wk=('median_weeks', 'first')
).reset_index()
summary['se'] = summary['std_change'] / np.sqrt(summary['count'])
summary['ci95'] = 1.96 * summary['se']

fig, ax = plt.subplots(figsize=(12, 7))
colors_tx = {'Solanezumab': '#3498db', 'Placebo': '#e74c3c'}

for tx in ['Placebo', 'Solanezumab']:
    tdf = summary[summary['TX'] == tx].sort_values('median_wk')
    ax.plot(tdf['median_wk'], tdf['mean_change'], 'o-', color=colors_tx[tx], linewidth=2, 
            markersize=5, label=tx)
    ax.fill_between(tdf['median_wk'], 
                    tdf['mean_change'] - tdf['ci95'], 
                    tdf['mean_change'] + tdf['ci95'],
                    color=colors_tx[tx], alpha=0.15)

ax.set_xlabel('Weeks from Randomization (Visit 006)')
ax.set_ylabel('PACC Change from Baseline')
ax.set_title('PACC Composite Score Over Time by Treatment Arm\n(Grouped by VISCODE, Baseline = Visit 006)')
ax.legend()
ax.axhline(y=0, color='gray', linestyle='-', alpha=0.3)
ax.axvline(x=240, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis4_pacc_trajectories.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Spline mixed model (approximates the published cLDA) ──
import statsmodels.formula.api as smf

# ── Crude endpoint comparison (t-test, weeks >= 220) ──
endpoint_df = pacc_rand[pacc_rand['weeks'] >= 220]
sol_end = endpoint_df[endpoint_df['TX'] == 'Solanezumab']['pacc_change']
pla_end = endpoint_df[endpoint_df['TX'] == 'Placebo']['pacc_change']
diff = sol_end.mean() - pla_end.mean()
t_stat, p_val = stats.ttest_ind(sol_end, pla_end)
se_diff = np.sqrt(sol_end.var()/len(sol_end) + pla_end.var()/len(pla_end))
ci_low, ci_high = diff - 1.96 * se_diff, diff + 1.96 * se_diff
print(f'Crude comparison (weeks >= 220): sol {sol_end.mean():.2f} (n={len(sol_end)}), '
      f'pla {pla_end.mean():.2f} (n={len(pla_end)}), '
      f'diff {diff:.2f} ({ci_low:.2f} to {ci_high:.2f}), p={p_val:.3f}')

# ── Mixed model: change ~ TX * ncs(weeks) + baseline PACC + randomization strata ──
# Strata: APOE e4 carriage, education >12 y. Site is not in OMOP (known limitation).
model_df = pacc_rand[['person_id', 'TX', 'weeks', 'pacc_change', 'pacc_baseline']].dropna().copy()
model_df = model_df[model_df['weeks'] > 0]
model_df['TX_code'] = (model_df['TX'] == 'Solanezumab').astype(int)
model_df['apoe_e4'] = model_df['person_id'].map(apoe_bl.set_index('person_id')['value_as_number'])
model_df['edu_gt12'] = (model_df['person_id'].map(edu_map) > 12).astype(int)
model_df = model_df.dropna(subset=['apoe_e4', 'pacc_baseline'])
print(f'Model dataset: {len(model_df)} obs, {model_df.person_id.nunique()} subjects')

pacc_time_knots = np.percentile(model_df['weeks'].values, [10, 50, 90])
_sp_pacc = rcs_basis(model_df['weeks'].values, pacc_time_knots)
model_df['sp1'], model_df['sp2'] = _sp_pacc[:, 0], _sp_pacc[:, 1]

full_result = smf.mixedlm(
    'pacc_change ~ TX_code * (sp1 + sp2) + pacc_baseline + apoe_e4 + edu_gt12',
    data=model_df, groups=model_df['person_id']).fit(reml=True)
print(full_result.summary().tables[1])

# ── Treatment effect at 240 weeks (delta method on the spline contrast) ──
_sp_240 = rcs_basis(np.array([240.0]), pacc_time_knots)
contrast = np.array([1, _sp_240[0, 0], _sp_240[0, 1]])
tx_params = ['TX_code', 'TX_code:sp1', 'TX_code:sp2']
tx_effect_full = contrast @ full_result.params[tx_params]
se_tx = np.sqrt(contrast @ full_result.cov_params().loc[tx_params, tx_params].values @ contrast)
ci_low_mmrm, ci_high_mmrm = tx_effect_full - 1.96 * se_tx, tx_effect_full + 1.96 * se_tx
p_mmrm = 2 * (1 - stats.norm.cdf(abs(tx_effect_full / se_tx)))

print(f'\n--- Treatment effect at 240 weeks ---')
print(f'  Spline mixed model:     {tx_effect_full:.2f} ({ci_low_mmrm:.2f} to {ci_high_mmrm:.2f}), p={p_mmrm:.3f}')
print(f'  Published (NEJM 2023): -0.30 (-0.82 to 0.22), p=0.26')
print(f'  Direction {"matches" if tx_effect_full < 0 else "flipped"} (solanezumab numerically worse); '
      f'within published CI: {"yes" if -0.82 <= tx_effect_full <= 0.22 else "no"}')

---
## Additional Replications (Analyses 6–13)

Remaining findings from the three papers:
- **Sperling (NEJM 2023)**: amyloid PET change, CFI / CDR-SB / ADL secondary endpoints, subgroups
- **Rissman (JPAD 2024)**: longitudinal p-tau217 by arm
- **Rentz (JPAD 2024)**: CDR box-score progression, CDR-SB by tertile

Plus two descriptive extensions with no published comparator in these papers: other plasma-biomarker AUROCs (Analysis 9) and a p-tau217 pre-screening scenario (Analysis 12).

Coverage notes: safety/ARIA outcomes (Sperling, Table 3) are not reproducible — the shared dataset has no adverse-event files. NEJM Fig. S4 (post-hoc PACC by amyloid tertile) is supplement-only with no values quoted in the main text, so it is not separately replicated.

In [ ]:
# ── Analysis 6: Amyloid PET Change by Treatment Arm (Sperling et al., Table 2) ──
# Published: amyloid rose in BOTH arms at 240 weeks: +11.6 CL (solanezumab) vs
# +19.3 CL (placebo); difference -7.7 CL (95% CI -10.4 to -5.1). Solanezumab
# slowed, but did not reverse, accumulation.

# ── Extract longitudinal amyloid PET composite SUVR ──
amy_long = get_measurements('AMYLOID\\|Florbetapir\\|Composite_Summary').copy()
amy_long['measurement_date'] = pd.to_datetime(amy_long['measurement_date'])
amy_long['centiloids'] = suvr_to_centiloids(amy_long['value_as_number'])

# Restrict to randomized subjects
amy_long_rand = amy_long[amy_long['person_id'].isin(randomized_ids)].copy()
amy_long_rand['TX'] = amy_long_rand['person_id'].map(tx_lookup)

# Baseline = earliest amyloid PET per person (imaging visits aren't visit 006)
amy_bl_rand = get_baseline(amy_long[amy_long['person_id'].isin(randomized_ids)])
amy_bl_vals = amy_bl_rand.set_index('person_id')['value_as_number'].rename('amy_bl_suvr')

amy_long_rand = amy_long_rand.merge(amy_bl_vals, on='person_id', how='inner')
amy_long_rand['cl_baseline'] = suvr_to_centiloids(amy_long_rand['amy_bl_suvr'])
amy_long_rand['cl_change'] = amy_long_rand['centiloids'] - amy_long_rand['cl_baseline']

# Weeks from randomization
amy_long_rand['rand_date'] = amy_long_rand['person_id'].map(V006_DATES)
amy_long_rand['weeks'] = (amy_long_rand['measurement_date'] - amy_long_rand['rand_date']).dt.days / 7
amy_long_rand = amy_long_rand.dropna(subset=['rand_date', 'cl_baseline'])

# Subjects with ≥1 post-baseline scan
post_bl = amy_long_rand[amy_long_rand['weeks'] > 4]
subj_counts = post_bl.groupby('person_id').size()
subj_2plus = set(subj_counts[subj_counts >= 1].index)  # ≥1 post-baseline
amy_long_rand = amy_long_rand[amy_long_rand['person_id'].isin(subj_2plus)]

print(f'Amyloid PET longitudinal (randomized): {len(amy_long_rand)} scans, '
      f'{amy_long_rand.person_id.nunique()} subjects')
print(f'  Solanezumab: {amy_long_rand[amy_long_rand.TX=="Solanezumab"].person_id.nunique()}')
print(f'  Placebo:     {amy_long_rand[amy_long_rand.TX=="Placebo"].person_id.nunique()}')

# ── Endpoint comparison at ~240 weeks ──
amy_endpoint = amy_long_rand[amy_long_rand['weeks'] >= 220].copy()
amy_sol = amy_endpoint[amy_endpoint['TX'] == 'Solanezumab']['cl_change']
amy_pla = amy_endpoint[amy_endpoint['TX'] == 'Placebo']['cl_change']

if len(amy_sol) > 5 and len(amy_pla) > 5:
    amy_diff = amy_sol.mean() - amy_pla.mean()
    amy_t, amy_p = stats.ttest_ind(amy_sol, amy_pla)
    amy_se = np.sqrt(amy_sol.var()/len(amy_sol) + amy_pla.var()/len(amy_pla))
    
    print(f'\n--- Amyloid PET Change from Baseline (≥220 weeks) ---')
    print(f'  Solanezumab: {amy_sol.mean():+.1f} CL (SD={amy_sol.std():.1f}, n={len(amy_sol)})')
    print(f'  Placebo:     {amy_pla.mean():+.1f} CL (SD={amy_pla.std():.1f}, n={len(amy_pla)})')
    print(f'  Difference:  {amy_diff:+.1f} CL (p={amy_p:.4f})')
    print(f'  Published:   Solanezumab=+11.6, Placebo=+19.3, Diff=-7.7 CL')
    print(f'  (Rissman et al. report the same wk-240 contrast as -7.6 CL, nominal p<0.001)')
    
    # Check direction: solanezumab should show LESS amyloid accumulation
    direction_ok = amy_sol.mean() < amy_pla.mean()
    print(f'  Direction: {"CORRECT" if direction_ok else "UNEXPECTED"} '
          f'(solanezumab {"less" if direction_ok else "more"} accumulation)')
else:
    # Few subjects at endpoint — use all post-baseline data
    print(f'\nFew subjects at ≥220 weeks. Using latest scan per subject.')
    latest_idx = amy_long_rand[amy_long_rand['weeks'] > 4].groupby('person_id')['weeks'].idxmax()
    amy_latest = amy_long_rand.loc[latest_idx]
    
    for arm in ['Solanezumab', 'Placebo']:
        arm_data = amy_latest[amy_latest['TX'] == arm]['cl_change']
        print(f'  {arm}: {arm_data.mean():+.1f} CL (SD={arm_data.std():.1f}, n={len(arm_data)}, '
              f'median {amy_latest[amy_latest["TX"]==arm]["weeks"].median():.0f} weeks)')

# ── Trajectory plot ──
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Solanezumab': '#e74c3c', 'Placebo': '#3498db'}
for arm in ['Solanezumab', 'Placebo']:
    arm_data = amy_long_rand[amy_long_rand['TX'] == arm]
    # Bin by ~50 week intervals for trajectory
    arm_data = arm_data.copy()
    arm_data['week_bin'] = (arm_data['weeks'] / 50).round() * 50
    traj = arm_data.groupby('week_bin')['cl_change'].agg(['mean', 'sem', 'count'])
    traj = traj[traj['count'] >= 10]  # require ≥10 subjects per bin
    ax.errorbar(traj.index, traj['mean'], yerr=1.96*traj['sem'],
                marker='o', capsize=3, label=f'{arm} (n={arm_data.person_id.nunique()})',
                color=colors[arm], linewidth=2)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Weeks from Randomization')
ax.set_ylabel('Change in Amyloid PET (Centiloids)')
ax.set_title('Amyloid PET Change by Treatment Arm\n(Sperling et al., NEJM 2023, Table 2)')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis6_amyloid_change.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Analysis 7: Secondary Endpoints (Sperling et al., Table 2) ──
# Published differences at 240 weeks (multiplicity-adjusted CIs; the hierarchical
# testing scheme stopped after the failed primary, so the paper reports NO p-values):
#   CFI combined (participant + partner, 0-30): diff  0.58 (95% CI -0.18 to 1.34)
#   CDR-SB:                                     diff  0.12 (-0.06 to 0.29)
#   ADL partner score (0-45):                   diff -0.61 (-1.44 to 0.23)
# Ours are crude endpoint t-tests; compare direction and rough magnitude only.

def endpoint_change_by_arm(meas, endpoint_name, published_diff):
    """Change-from-baseline arm comparison at weeks >= 200."""
    meas = meas.copy()
    meas['measurement_date'] = pd.to_datetime(meas['measurement_date'])
    meas = meas[meas['person_id'].isin(randomized_ids)]
    meas['TX'] = meas['person_id'].map(tx_lookup)

    bl = get_baseline_v006(meas)
    if len(bl) == 0:
        bl = get_baseline(meas)
    bl_vals = bl.set_index('person_id')['value_as_number'].rename('bl_val')
    meas = meas.merge(bl_vals, on='person_id', how='inner')
    meas['change'] = meas['value_as_number'] - meas['bl_val']
    meas['weeks'] = (meas['measurement_date'] - meas['person_id'].map(V006_DATES)).dt.days / 7
    meas = meas.dropna(subset=['weeks', 'bl_val'])

    late = meas[meas['weeks'] >= 200]
    sol = late[late['TX'] == 'Solanezumab']['change']
    pla = late[late['TX'] == 'Placebo']['change']
    diff = sol.mean() - pla.mean()
    _, p = stats.ttest_ind(sol, pla)
    return {'endpoint': endpoint_name, 'sol': sol.mean(), 'pla': pla.mean(),
            'n_sol': len(sol), 'n_pla': len(pla), 'diff': diff, 'p': p,
            'published_diff': published_diff,
            'direction_match': diff * published_diff > 0,
            'median_weeks': late['weeks'].median()}

# CFI combined = participant total + study-partner total (the paper's endpoint, 0-30)
cfi_pt = get_measurements('CFI:CFIPTTOTAL', exact=True)
cfi_sp = get_measurements('CFISP:CFSPTTOTAL', exact=True)
cfi = cfi_pt.merge(cfi_sp[['person_id', 'measurement_date', 'value_as_number']],
                   on=['person_id', 'measurement_date'], suffixes=('', '_sp'))
cfi['value_as_number'] = cfi['value_as_number'] + cfi['value_as_number_sp']

secondary_results = {}
for meas, name, pub in [
    (cfi, 'CFI combined', 0.58),
    (get_measurements('CDR:CDSOB'), 'CDR-SB', 0.12),
    (get_measurements('ADLPQSP:AISCORE', exact=True), 'ADL partner score', -0.61),
]:
    res = endpoint_change_by_arm(meas, name, pub)
    secondary_results[name] = res
    tag = 'direction matches' if res['direction_match'] else 'direction differs'
    print(f"{name:<20} OMOP diff {res['diff']:+.3f} (p={res['p']:.3f})   "
          f"published {res['published_diff']:+.2f}   [{tag}]")
    print(f"  sol {res['sol']:+.3f} (n={res['n_sol']}), pla {res['pla']:+.3f} "
          f"(n={res['n_pla']}), median {res['median_weeks']:.0f} wks")

print('\nPublished diffs are from adjusted longitudinal models with no reported')
print('p-values; ours are crude t-tests, so direction is the comparison that matters.')

In [ ]:
# ── CDR-Global progression by arm (Sperling et al., Table 2) ──
# Published KM probability of progression by week 240: solanezumab 0.35 vs
# placebo 0.32 (33.4% overall). NEJM counts a non-zero CDR-G at two consecutive
# visits OR at the final visit (broader than the two-consecutive-only definition
# Rentz uses in Analysis 3).

nejm_recs = []
for pid, g in cdr_global[cdr_global['person_id'].isin(randomized_ids)].groupby('person_id'):
    g = g.sort_values('measurement_date')
    if g['value_as_number'].iloc[0] != 0:
        continue  # baseline CDR-G must be 0
    fu = g.iloc[1:]
    if len(fu) == 0:
        continue
    v = fu['value_as_number'].values
    dts = fu['measurement_date'].values
    ev_date = next((dts[i] for i in range(len(v) - 1) if v[i] > 0 and v[i + 1] > 0), None)
    if ev_date is None and v[-1] > 0:
        ev_date = dts[-1]
    t0 = g['measurement_date'].iloc[0]
    end = ev_date if ev_date is not None else dts[-1]
    t = (pd.Timestamp(end) - t0).days / 7
    if t > 0:
        nejm_recs.append({'person_id': pid, 't': t, 'e': int(ev_date is not None)})

nejm_df = pd.DataFrame(nejm_recs)
nejm_df['TX'] = nejm_df['person_id'].map(tx_lookup)

published_cdr_arm = {'Solanezumab': 35, 'Placebo': 32, 'Overall': 33.4}
cdr_arm_km = {}
for arm in ['Solanezumab', 'Placebo', 'Overall']:
    d = nejm_df if arm == 'Overall' else nejm_df[nejm_df['TX'] == arm]
    km = kaplan_meier(d['t'].values, d['e'].values)
    rate = (1 - km_survival_at(km, 240)) * 100
    cdr_arm_km[arm] = rate
    print(f'  {arm:12s} n={len(d):4d}  KM progression by wk 240: {rate:.1f}%  '
          f'(published {published_cdr_arm[arm]}%)')

In [ ]:
# ── Analysis 8: PACC Subgroup Analyses (Sperling et al., supplementary Fig. S2) ──
# Paper: "Prespecified subgroup analyses showed similar results" -- one sentence,
# no per-subgroup estimates in the main text. Our subgroup set approximates it.

# Use pacc_rand from Analysis 4 — already has TX, weeks, pacc_change, pacc_baseline

# ── Define subgroups ──
_p = person.set_index('person_id')
subgroup_data = pacc_rand[pacc_rand['weeks'] >= 200].copy()
if len(subgroup_data) < 50:
    subgroup_data = pacc_rand[pacc_rand['weeks'] >= 150].copy()
if len(subgroup_data) < 50:
    # Take latest per person
    idx = pacc_rand[pacc_rand['weeks'] > 0].groupby('person_id')['weeks'].idxmax()
    subgroup_data = pacc_rand.loc[idx]

subgroup_data['age'] = subgroup_data['person_id'].map(lambda p: 2020 - _p.loc[p, 'year_of_birth'])
subgroup_data['sex'] = subgroup_data['person_id'].map(lambda p: 'Female' if _p.loc[p, 'gender_concept_id'] == 8532 else 'Male')
subgroup_data['apoe_e4'] = subgroup_data['person_id'].map(
    lambda p: apoe_bl.set_index('person_id')['value_as_number'].get(p, np.nan))
subgroup_data['bl_amyloid'] = subgroup_data['person_id'].map(
    amy_bl_pos.set_index('person_id')['centiloids'])

# Create subgroup categories (matching published)
subgroup_data['age_group'] = np.where(subgroup_data['age'] <= 72, '≤72', '>72')
subgroup_data['apoe_group'] = np.where(subgroup_data['apoe_e4'] == 1, 'APOE4+', 'APOE4−')
subgroup_data['amyloid_group'] = np.where(
    subgroup_data['bl_amyloid'] <= subgroup_data['bl_amyloid'].median(),
    'Below median', 'Above median')

# ── Compute treatment effect per subgroup ──
subgroups = {
    'Overall': subgroup_data,
    'Age ≤72': subgroup_data[subgroup_data['age_group'] == '≤72'],
    'Age >72': subgroup_data[subgroup_data['age_group'] == '>72'],
    'Female': subgroup_data[subgroup_data['sex'] == 'Female'],
    'Male': subgroup_data[subgroup_data['sex'] == 'Male'],
    'APOE4 carrier': subgroup_data[subgroup_data['apoe_group'] == 'APOE4+'],
    'APOE4 non-carrier': subgroup_data[subgroup_data['apoe_group'] == 'APOE4−'],
    'Amyloid below median': subgroup_data[subgroup_data['amyloid_group'] == 'Below median'],
    'Amyloid above median': subgroup_data[subgroup_data['amyloid_group'] == 'Above median'],
}

sg_results = []
for name, sdf in subgroups.items():
    sol = sdf[sdf['TX'] == 'Solanezumab']['pacc_change']
    pla = sdf[sdf['TX'] == 'Placebo']['pacc_change']
    if len(sol) >= 5 and len(pla) >= 5:
        d = sol.mean() - pla.mean()
        se = np.sqrt(sol.var()/len(sol) + pla.var()/len(pla))
        sg_results.append({
            'Subgroup': name, 'N_sol': len(sol), 'N_pla': len(pla),
            'Diff': d, 'CI_low': d - 1.96*se, 'CI_high': d + 1.96*se,
            'SE': se,
        })

sg_df = pd.DataFrame(sg_results)
print('══════════════════════════════════════════════════════════════')
print('PACC Subgroup Analyses (Sperling et al., NEJM 2023, Fig. S2)')
print('══════════════════════════════════════════════════════════════')
print(f'{"Subgroup":<25} {"N(Sol)":<8} {"N(Pla)":<8} {"Diff":<8} {"95% CI":<20}')
print('─' * 69)
for _, row in sg_df.iterrows():
    print(f'{row["Subgroup"]:<25} {row["N_sol"]:<8.0f} {row["N_pla"]:<8.0f} '
          f'{row["Diff"]:<+8.2f} ({row["CI_low"]:+.2f} to {row["CI_high"]:+.2f})')

# ── Forest plot ──
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(sg_df) - 1, -1, -1)
colors = ['#2c3e50' if r['Subgroup'] == 'Overall' else '#7f8c8d' for _, r in sg_df.iterrows()]

for i, (_, row) in enumerate(sg_df.iterrows()):
    y = list(y_pos)[i]
    c = colors[i]
    ax.errorbar(row['Diff'], y, xerr=[[row['Diff']-row['CI_low']], [row['CI_high']-row['Diff']]],
                fmt='o' if row['Subgroup'] == 'Overall' else 's', 
                color=c, capsize=4, markersize=8 if row['Subgroup'] == 'Overall' else 6)

ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(sg_df['Subgroup'].tolist())
ax.set_xlabel('PACC Difference (Solanezumab − Placebo)')
ax.set_title('PACC Treatment Effect by Subgroup\n(Sperling et al., NEJM 2023)')

# Add "Favors Solanezumab" / "Favors Placebo" labels
xlim = ax.get_xlim()
ax.text(xlim[0] * 0.9, -1.2, '← Favors Solanezumab', fontsize=8, ha='left', style='italic')
ax.text(xlim[1] * 0.9, -1.2, 'Favors Placebo →', fontsize=8, ha='right', style='italic')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis8_pacc_subgroups.png', bbox_inches='tight')
plt.show()

print('\nPaper reports only that subgroup results were \"similar\" to the overall null.')
all_contain_zero = all((row['CI_low'] <= 0 <= row['CI_high']) for _, row in sg_df.iterrows())
print(f'Our finding: All CIs contain zero = {all_contain_zero} (consistent with published null result)')

In [ ]:
# ── Analysis 9: Other Plasma Biomarker AUROCs (descriptive extension) ──
# Rissman et al. report AUROCs only for p-tau217 (0.87 for >=20 CL, Fig. 1C).
# The other assays here (ELISA Abeta42/40, Roche GFAP / NF-L) exist in the A4
# external files but have no published AUROC in our three source papers, so
# everything except p-tau217 is descriptive.

ab4240_bl = get_baseline(get_measurements('AB:FP42/FP40\\|PLASMA\\|ELISA'))[
    ['person_id', 'value_as_number']].rename(columns={'value_as_number': 'ab4240'})
gfap_bl = get_baseline(get_measurements('ROCHE:GFAP'))[
    ['person_id', 'value_as_number']].rename(columns={'value_as_number': 'gfap'})
nfl_bl = get_baseline(get_measurements('ROCHE:NF-L'))[
    ['person_id', 'value_as_number']].rename(columns={'value_as_number': 'nfl'})

bio_df = amyloid_pet_bl[['person_id', 'centiloids']].copy()
for bl in [ptau217_bl, ab4240_bl, gfap_bl, nfl_bl]:
    bio_df = bio_df.merge(bl, on='person_id', how='left')

y_true_all = (bio_df['centiloids'] >= 20).astype(int)

# column: (label, published AUROC if any, invert score (low Abeta42/40 = positive))
markers = {
    'ptau217': ('P-tau217', 0.87, False),
    'ab4240': ('Aβ42/40 (ELISA)', None, True),
    'gfap': ('GFAP (Roche)', None, False),
    'nfl': ('NF-L (Roche)', None, False),
}

biomarker_rocs = {}
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (col, (name, pub, inv)) in zip(axes.flatten(), markers.items()):
    mask = bio_df[col].notna()
    y_true = y_true_all[mask]
    y_score = -bio_df.loc[mask, col] if inv else bio_df.loc[mask, col]
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    biomarker_rocs[col] = auc
    ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'OMOP AUROC = {auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    if pub is not None:
        ax.axhline(y=pub, color='r', linestyle=':', alpha=0.5, label=f'Published = {pub:.2f}')
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.set_title(f'{name} → Amyloid ≥20 CL (n={mask.sum()})')
    ax.legend(loc='lower right', fontsize=9)

plt.suptitle('Plasma Biomarkers → Amyloid PET ≥20 CL\n'
             '(published comparator exists for p-tau217 only)', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis9_biomarker_rocs.png', bbox_inches='tight')
plt.show()

# ── Multi-marker panel (cross-validated logistic regression) ──
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

multi_mask = bio_df[list(markers)].notna().all(axis=1)
panel = SkPipeline([('scaler', StandardScaler()),
                    ('clf', LogisticRegression(max_iter=1000))])
cv_auc = cross_val_score(panel, bio_df.loc[multi_mask, list(markers)].values,
                         y_true_all[multi_mask].values, scoring='roc_auc',
                         cv=StratifiedKFold(5, shuffle=True, random_state=42))
biomarker_rocs['multi_panel'] = cv_auc.mean()
print(f'Multi-marker panel (5-fold CV): AUROC={cv_auc.mean():.3f} (±{cv_auc.std():.3f}), '
      f'n={multi_mask.sum()} with all 4 markers')

print('\n--- Biomarker AUROCs for amyloid PET >=20 CL ---')
for col, (name, pub, _) in markers.items():
    pub_str = f'published {pub:.2f} (Rissman)' if pub else 'no published A4 comparator'
    print(f'  {name:<18} {biomarker_rocs[col]:.3f}   {pub_str}')
print(f'  {"Multi-panel (CV)":<18} {biomarker_rocs["multi_panel"]:.3f}   no published A4 comparator')

In [ ]:
# ── Analysis 10: Longitudinal P-tau217 by Treatment Arm (Rissman et al.) ──
# Published: p-tau217 rose in both arms (assayed at baseline, week 12, week 240).
# Week-240 arm difference -0.018 U/mL (95% CI -0.055 to 0.019; nominal p=0.352):
# numerically less accumulation on solanezumab, NOT statistically significant.

ptau_long = get_measurements('PTAU217').copy()
ptau_long['measurement_date'] = pd.to_datetime(ptau_long['measurement_date'])

# Restrict to randomized
ptau_long_rand = ptau_long[ptau_long['person_id'].isin(randomized_ids)].copy()
ptau_long_rand['TX'] = ptau_long_rand['person_id'].map(tx_lookup)

# Baseline at visit 006
ptau_bl_v006 = get_baseline_v006(ptau_long)
ptau_bl_v006 = ptau_bl_v006[ptau_bl_v006['person_id'].isin(randomized_ids)]
ptau_bl_map = ptau_bl_v006.set_index('person_id')['value_as_number'].rename('ptau_bl')

ptau_long_rand = ptau_long_rand.merge(ptau_bl_map, on='person_id', how='inner')
ptau_long_rand['ptau_change'] = ptau_long_rand['value_as_number'] - ptau_long_rand['ptau_bl']
ptau_long_rand['ptau_pct_change'] = (ptau_long_rand['ptau_change'] / ptau_long_rand['ptau_bl']) * 100

# Weeks from randomization  
ptau_long_rand['rand_date'] = ptau_long_rand['person_id'].map(V006_DATES)
ptau_long_rand['weeks'] = (ptau_long_rand['measurement_date'] - ptau_long_rand['rand_date']).dt.days / 7
ptau_long_rand = ptau_long_rand.dropna(subset=['rand_date', 'ptau_bl'])

n_total = ptau_long_rand.person_id.nunique()
n_sol = ptau_long_rand[ptau_long_rand.TX == 'Solanezumab'].person_id.nunique()
n_pla = ptau_long_rand[ptau_long_rand.TX == 'Placebo'].person_id.nunique()
print(f'Longitudinal P-tau217 (randomized): {len(ptau_long_rand)} records, {n_total} subjects')
print(f'  Solanezumab: {n_sol}, Placebo: {n_pla}')

# ── P-tau217 trajectory plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, ylabel in [
    (axes[0], 'ptau_change', 'P-tau217 Change (absolute)'),
    (axes[1], 'ptau_pct_change', 'P-tau217 Change (%)')
]:
    for arm, color in [('Solanezumab', '#e74c3c'), ('Placebo', '#3498db')]:
        arm_data = ptau_long_rand[ptau_long_rand['TX'] == arm].copy()
        arm_data['week_bin'] = (arm_data['weeks'] / 50).round() * 50
        traj = arm_data.groupby('week_bin')[metric].agg(['mean', 'sem', 'count'])
        traj = traj[traj['count'] >= 10]
        ax.errorbar(traj.index, traj['mean'], yerr=1.96*traj['sem'],
                    marker='o', capsize=3, color=color, linewidth=2,
                    label=f'{arm} (n={arm_data.person_id.nunique()})')
    
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Weeks from Randomization')
    ax.set_ylabel(ylabel)
    ax.legend()

axes[0].set_title('P-tau217 Absolute Change by Arm')
axes[1].set_title('P-tau217 % Change by Arm')
plt.suptitle('Longitudinal P-tau217 by Treatment\n(Rissman et al., JPAD 2024)', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis10_ptau217_longitudinal.png', bbox_inches='tight')
plt.show()

# ── Endpoint comparison ──
ptau_late = ptau_long_rand[ptau_long_rand['weeks'] >= 200]
if ptau_late.person_id.nunique() < 20:
    ptau_late = ptau_long_rand[ptau_long_rand['weeks'] > 0]
    idx = ptau_late.groupby('person_id')['weeks'].idxmax()
    ptau_late = ptau_late.loc[idx]

for metric_name, col in [('Absolute change', 'ptau_change'), ('% change', 'ptau_pct_change')]:
    sol = ptau_late[ptau_late['TX'] == 'Solanezumab'][col]
    pla = ptau_late[ptau_late['TX'] == 'Placebo'][col]
    if len(sol) > 5 and len(pla) > 5:
        d = sol.mean() - pla.mean()
        t, p = stats.ttest_ind(sol, pla)
        print(f'\n{metric_name}:')
        print(f'  Solanezumab: {sol.mean():+.3f} (SD={sol.std():.3f}, n={len(sol)})')
        print(f'  Placebo:     {pla.mean():+.3f} (SD={pla.std():.3f}, n={len(pla)})')
        print(f'  Difference:  {d:+.3f} (p={p:.4f})')
        print(f'  Direction: {"consistent" if d < 0 else "differs"} vs the published numeric trend (ns in paper, p=0.352)')

In [ ]:
# ── Analysis 11: CDR Box Score Progression by Amyloid Tertile (Rentz et al.) ──
# Published week-240 memory-box progression by tertile: 18% / 34% / 41%; memory
# and judgment showed the clearest amyloid dose-response of the 6 boxes.

cdr_domains = {
    'Memory': 'CDR:MEMORY',
    'Orientation': 'CDR:ORIENT', 
    'Judgment': 'CDR:JUDGE',
    'Community': 'CDR:COMMUN',
    'Home': 'CDR:HOME',
    'Personal Care': 'CDR:CARE',
}

# Use the same analysis population as the main CDR analysis (cell 16):
# randomized, amyloid-positive (≥20 CL), both arms, baseline CDR-G=0
# persons_with_both and amy_bl_pos already defined

box_progression = {}

for domain_name, source_prefix in cdr_domains.items():
    # Get all records matching this domain (SP=1.0, in-person visits for consistency)
    domain_data = measurement[
        measurement['measurement_source_value'].str.startswith(source_prefix + '|', na=False)
    ].copy()
    domain_data['measurement_date'] = pd.to_datetime(domain_data['measurement_date'])
    
    # Filter to analysis population
    domain_data = domain_data[domain_data['person_id'].isin(persons_with_both)]
    
    if len(domain_data) == 0:
        print(f'{domain_name}: no data')
        continue
    
    # Baseline (earliest) per person
    bl_idx = domain_data.groupby('person_id')['measurement_date'].idxmin()
    bl_data = domain_data.loc[bl_idx]
    bl_vals = bl_data.set_index('person_id')['value_as_number']
    
    # Restrict to those with baseline = 0 in this domain
    bl_zero = set(bl_vals[bl_vals == 0].index)
    
    # Follow-up data for those with baseline=0
    fu_data = domain_data[
        (domain_data['person_id'].isin(bl_zero)) & 
        (domain_data['measurement_date'] > domain_data['person_id'].map(
            bl_data.set_index('person_id')['measurement_date']))
    ].sort_values(['person_id', 'measurement_date'])
    
    # Track confirmed progression (two consecutive > 0)
    prog_persons = set()
    for pid in bl_zero:
        pid_fu = fu_data[fu_data['person_id'] == pid]
        vals = pid_fu['value_as_number'].values
        for i in range(len(vals) - 1):
            if vals[i] > 0 and vals[i + 1] > 0:
                prog_persons.add(pid)
                break
    
    # Rates by tertile
    domain_rates = {}
    for tname in ['Low', 'Intermediate', 'High']:
        t_persons = set(amy_bl_pos[amy_bl_pos['tertile'] == tname]['person_id'])
        eligible = bl_zero & t_persons
        progressed = prog_persons & eligible
        rate = len(progressed) / len(eligible) * 100 if len(eligible) > 0 else 0
        domain_rates[tname] = {'rate': rate, 'n_eligible': len(eligible), 
                                'n_progressed': len(progressed)}
    
    box_progression[domain_name] = domain_rates

# ── Display results ──
print('══════════════════════════════════════════════════════════════')
print('CDR Box Score Progression by Amyloid Tertile (Rentz et al.)')
print('══════════════════════════════════════════════════════════════')
print(f'Population: Randomized, amyloid+, CDR-G=0 at baseline, both arms')
print(f'Progression: Two consecutive visits with domain score > 0')
print()
print(f'{"Domain":<18} {"Low":<12} {"Intermediate":<14} {"High":<12} {"Gradient":<10}')
print('─' * 66)

for domain_name in cdr_domains:
    if domain_name in box_progression:
        rates = box_progression[domain_name]
        low_r = rates['Low']['rate']
        mid_r = rates['Intermediate']['rate']
        high_r = rates['High']['rate']
        gradient = 'YES' if low_r < mid_r < high_r else 'partial'
        print(f'{domain_name:<18} {low_r:>5.1f}% (n={rates["Low"]["n_eligible"]:<3}) '
              f'{mid_r:>5.1f}% (n={rates["Intermediate"]["n_eligible"]:<3}) '
              f'{high_r:>5.1f}% (n={rates["High"]["n_eligible"]:<3}) {gradient}')

# ── Grouped bar chart ──
fig, ax = plt.subplots(figsize=(12, 6))
domains = list(box_progression.keys())
x = np.arange(len(domains))
width = 0.25
colors_tertile = {'Low': '#2ecc71', 'Intermediate': '#f39c12', 'High': '#e74c3c'}

for i, tname in enumerate(['Low', 'Intermediate', 'High']):
    rates = [box_progression[d][tname]['rate'] for d in domains]
    ax.bar(x + (i - 1) * width, rates, width, label=f'{tname} amyloid', 
           color=colors_tertile[tname], alpha=0.85)

ax.set_xlabel('CDR Domain')
ax.set_ylabel('Confirmed Progression Rate (%)')
ax.set_title('CDR Box Score Progression by Amyloid Tertile\n(Rentz et al., JPAD 2024)')
ax.set_xticks(x)
ax.set_xticklabels(domains, rotation=15, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis11_cdr_boxscores.png', bbox_inches='tight')
plt.show()

published_box = {  # Rentz et al., modeled wk-240 progression by tertile (Low/Int/High)
    'Memory': '18/34/41%', 'Orientation': '10/18/21%', 'Judgment': '17/28/32%',
    'Community': '0/11/16%', 'Home': '4/13/18%', 'Personal Care': '≤1% all tertiles',
}
print('\nOurs vs published (Rentz, modeled wk-240 rates), Low/Int/High:')
for dom, pub in published_box.items():
    if dom in box_progression:
        r = box_progression[dom]
        ours = '/'.join(f'{r[t]["rate"]:.0f}' for t in ['Low', 'Intermediate', 'High'])
        print(f'  {dom:<14} ours {ours}%   published {pub}')
print('Paper: memory and judgment showed the clearest amyloid dose-response.')

In [ ]:
# ── Analysis 12: P-tau217 as a Pre-Screening Tool (original scenario analysis) ──
# Rissman et al. conclude only that FUTURE analyses will assess whether p-tau217
# can reduce screening burden; the paper reports no thresholds or savings numbers.
# This cell works the scenario out in the OMOP data. No published comparator.

# ── Screen failure analysis ──
# In A4, all subjects got amyloid PET; amyloid-negative were excluded from randomization
# Scenario: What if P-tau217 was used as pre-screen before PET?

# Use roc_df from Analysis 2 (has ptau217 and centiloids for matched subjects)
print('══════════════════════════════════════════════════════════════')
print('P-tau217 as Pre-Screening Tool (scenario analysis)')
print('══════════════════════════════════════════════════════════════')
print(f'Scenario: Use P-tau217 to pre-screen before amyloid PET')
print(f'Goal: Reduce screen failures (amyloid-negative subjects requiring PET)')

# Published used ≥20 CL as amyloid-positive threshold
y_true = (roc_df['centiloids'] >= 20).astype(int)
total = len(roc_df)
n_pos = y_true.sum()
n_neg = total - n_pos
base_pos_rate = n_pos / total * 100

print(f'\nBaseline (no pre-screening):')
print(f'  Subjects screened: {total}')
print(f'  Amyloid-positive: {n_pos} ({base_pos_rate:.1f}%)')
print(f'  Screen failures: {n_neg} ({100-base_pos_rate:.1f}%)')

# Test various P-tau217 thresholds
print(f'\n{"P-tau217 Threshold":<20} {"Referred to PET":<16} {"True Pos":<10} {"Missed":<10} '
      f'{"PPV":<8} {"Sensitivity":<12} {"PET Savings":<12}')
print('─' * 88)

enrichment_results = []
for pct in [25, 33, 50, 67, 75]:
    threshold = np.percentile(roc_df['ptau217'], pct)
    selected = roc_df['ptau217'] >= threshold
    n_selected = selected.sum()
    true_pos = (selected & (y_true == 1)).sum()
    missed = ((~selected) & (y_true == 1)).sum()
    ppv = true_pos / n_selected * 100 if n_selected > 0 else 0
    sensitivity = true_pos / n_pos * 100 if n_pos > 0 else 0
    pet_savings = (1 - n_selected / total) * 100
    
    enrichment_results.append({
        'percentile': pct, 'threshold': threshold,
        'n_selected': n_selected, 'true_pos': true_pos,
        'missed': missed, 'ppv': ppv, 'sensitivity': sensitivity,
        'pet_savings': pet_savings
    })
    
    print(f'≥ p{pct} ({threshold:.2f}){"":<6} {n_selected:<16} {true_pos:<10} {missed:<10} '
          f'{ppv:<8.1f} {sensitivity:<12.1f} {pet_savings:<12.1f}%')

hi_sens = [r for r in enrichment_results if r['sensitivity'] >= 95]
if hi_sens:
    r = hi_sens[-1]
    print(f'\nHighest threshold keeping >=95% sensitivity: p{r["percentile"]} '
          f'({r["sensitivity"]:.0f}% sensitivity, {r["pet_savings"]:.0f}% of PET scans avoided)')

# ── Enrichment visualization ──
fig, ax1 = plt.subplots(figsize=(8, 5))
pcts = [r['percentile'] for r in enrichment_results]
sensitivities = [r['sensitivity'] for r in enrichment_results]
ppvs = [r['ppv'] for r in enrichment_results]
savings = [r['pet_savings'] for r in enrichment_results]

ax1.plot(pcts, sensitivities, 'b-o', linewidth=2, label='Sensitivity (%)')
ax1.plot(pcts, ppvs, 'r-s', linewidth=2, label='PPV (%)')
ax1.set_xlabel('P-tau217 Percentile Threshold')
ax1.set_ylabel('Performance (%)')
ax1.legend(loc='center left')
ax1.set_ylim(0, 105)

ax2 = ax1.twinx()
ax2.bar(pcts, savings, width=4, alpha=0.2, color='green', label='PET Savings (%)')
ax2.set_ylabel('PET Scans Avoided (%)', color='green')
ax2.tick_params(axis='y', labelcolor='green')
ax2.set_ylim(0, 105)

plt.title('P-tau217 Pre-Screening Trade-offs\n(scenario analysis, no published comparator)')
fig.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis12_enrichment.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Analysis 13: CDR-SB Change by Amyloid Tertile (Rentz et al.) ──
# Published (both arms combined, modeled means at week 240): 0.33 / 0.72 / 1.11
# by ascending tertile. The per-arm split below is our extension; the paper
# reports combined arms only.

cdrsb_all = measurement[
    measurement['measurement_source_value'].str.contains('CDR:CDSOB', na=False)
].copy()
cdrsb_all['measurement_date'] = pd.to_datetime(cdrsb_all['measurement_date'])

# Restrict to the Analysis 3 population (randomized, amyloid+, CDR-G=0 at baseline)
cdrsb_pop = cdrsb_all[cdrsb_all['person_id'].isin(persons_with_both)].copy()
cdrsb_pop['TX'] = cdrsb_pop['person_id'].map(tx_lookup)

# Change from earliest CDR-SB per person
cdrsb_bl_idx = cdrsb_pop.groupby('person_id')['measurement_date'].idxmin()
cdrsb_bl = cdrsb_pop.loc[cdrsb_bl_idx].set_index('person_id')['value_as_number'].rename('cdrsb_bl')
cdrsb_bl_dates = cdrsb_pop.loc[cdrsb_bl_idx].set_index('person_id')['measurement_date']

cdrsb_pop = cdrsb_pop.merge(cdrsb_bl, on='person_id', how='inner')
cdrsb_pop['cdrsb_change'] = cdrsb_pop['value_as_number'] - cdrsb_pop['cdrsb_bl']
cdrsb_pop = cdrsb_pop.merge(amy_bl_pos[['person_id', 'tertile']], on='person_id', how='inner')
cdrsb_pop['weeks'] = (cdrsb_pop['measurement_date']
                      - cdrsb_pop['person_id'].map(cdrsb_bl_dates)).dt.days / 7
cdrsb_pop = cdrsb_pop[cdrsb_pop['weeks'] > 0]
print(f'CDR-SB longitudinal: {len(cdrsb_pop)} records, {cdrsb_pop.person_id.nunique()} subjects')

# ── Latest CDR-SB change: combined arms vs published, then the per-arm extension ──
latest_idx = cdrsb_pop.groupby('person_id')['weeks'].idxmax()
cdrsb_latest = cdrsb_pop.loc[latest_idx]

print('\nCombined arms, latest change by tertile (published modeled wk-240 means: 0.33/0.72/1.11):')
for tname in ['Low', 'Intermediate', 'High']:
    s = cdrsb_latest[cdrsb_latest['tertile'] == tname]['cdrsb_change']
    print(f'  {tname:<13} {s.mean():+.2f} (n={len(s)})')

print('\nPer-arm split (our extension; not reported in the paper):')
print(f'{"Tertile":<15} {"Arm":<15} {"N":<6} {"Mean change":<14} {"SD":<8}')
for tname in ['Low', 'Intermediate', 'High']:
    for arm in ['Placebo', 'Solanezumab']:
        s = cdrsb_latest[(cdrsb_latest['tertile'] == tname) & (cdrsb_latest['TX'] == arm)]
        if len(s) > 0:
            print(f'{tname:<15} {arm:<15} {len(s):<6} {s["cdrsb_change"].mean():<+14.3f} '
                  f'{s["cdrsb_change"].std():<8.3f}')

# ── Trajectories by tertile ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
colors = {'Solanezumab': '#e74c3c', 'Placebo': '#3498db'}
for ax, tname in zip(axes, ['Low', 'Intermediate', 'High']):
    t_data = cdrsb_pop[cdrsb_pop['tertile'] == tname]
    for arm, color in colors.items():
        arm_data = t_data[t_data['TX'] == arm].copy()
        arm_data['week_bin'] = (arm_data['weeks'] / 50).round() * 50
        traj = arm_data.groupby('week_bin')['cdrsb_change'].agg(['mean', 'sem', 'count'])
        traj = traj[traj['count'] >= 5]
        ax.errorbar(traj.index, traj['mean'], yerr=1.96 * traj['sem'],
                    marker='o', capsize=3, color=color, linewidth=2,
                    label=f'{arm} (n={arm_data.person_id.nunique()})')
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Weeks')
    ax.set_title(f'{tname} Amyloid')
    ax.legend(fontsize=8)

axes[0].set_ylabel('CDR-SB Change from Baseline')
plt.suptitle('CDR-SB Change by Amyloid Tertile and Treatment\n(Rentz et al., JPAD 2024)', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis13_cdrsb_tertile.png', bbox_inches='tight')
plt.show()

---
## Analysis 5: Novel — Multi-Modal Prediction of Cognitive Decline

**Goal**: Demonstrate that the OMOP CDM's integrated structure enables multi-domain prediction that would be difficult without the CDM. We predict clinically meaningful cognitive decline using features from ALL OMOP tables simultaneously.

**Prediction target**: PACC decrease >1 SD from baseline at last follow-up.

**Key question**: Does multi-modal prediction outperform single-domain prediction?

In [ ]:
# ── Define outcome: PACC decline > 1 SD ──
pacc_long = get_measurements('PACC:PACC.raw', exact=True).copy()
pacc_long['measurement_date'] = pd.to_datetime(pacc_long['measurement_date'])

# Baseline PACC
pacc_bl_all = get_baseline(pacc_long)
pacc_bl_map = pacc_bl_all.set_index('person_id')['value_as_number'].rename('pacc_bl')
pacc_bl_date_map = pacc_bl_all.set_index('person_id')['measurement_date'].rename('pacc_bl_date')

# Last PACC per person
idx_last = pacc_long.groupby('person_id')['measurement_date'].idxmax()
pacc_last = pacc_long.loc[idx_last].set_index('person_id')['value_as_number'].rename('pacc_last')

# Change
outcome = pd.DataFrame({'pacc_bl': pacc_bl_map, 'pacc_last': pacc_last}).dropna()
outcome['pacc_change'] = outcome['pacc_last'] - outcome['pacc_bl']

# Threshold: decline > 1 SD of baseline PACC distribution
decline_threshold = -1 * outcome['pacc_bl'].std()
outcome['declined'] = (outcome['pacc_change'] < decline_threshold).astype(int)

print(f'Subjects with longitudinal PACC: {len(outcome)}')
print(f'Decline threshold: PACC change < {decline_threshold:.2f}')
print(f'Declined: {outcome["declined"].sum()} ({100*outcome["declined"].mean():.1f}%)')
print(f'Stable:   {(1-outcome["declined"]).sum()} ({100*(1-outcome["declined"].mean()):.1f}%)')

In [ ]:
# ── Feature Extraction from ALL OMOP Tables ──
features = pd.DataFrame(index=outcome.index)
features['declined'] = outcome['declined']

# --- PERSON table features ---
person_idx = person.set_index('person_id')
features['age'] = 2020 - person_idx.reindex(features.index)['year_of_birth']
features['female'] = (person_idx.reindex(features.index)['gender_concept_id'] == 8532).astype(int)

# Race: simplified to White vs other
# race_concept_id 8527 = White
features['white'] = (person_idx.reindex(features.index)['race_concept_id'] == 8527).astype(int)

print(f'PERSON features: age, female, white')

# --- MEASUREMENT features (baseline values) ---
def add_baseline_measurement(features_df, source_pattern, feature_name, exact=False):
    """Extract baseline measurement and add as feature column."""
    m = get_measurements(source_pattern, exact=exact)
    if len(m) == 0:
        print(f'  {feature_name}: 0 records found (pattern: {source_pattern})')
        return features_df
    m_bl = get_baseline(m)
    m_map = m_bl.set_index('person_id')['value_as_number']
    features_df[feature_name] = m_map.reindex(features_df.index)
    n_avail = features_df[feature_name].notna().sum()
    print(f'  {feature_name}: {n_avail} subjects')
    return features_df

print('\nMEASUREMENT features (baseline):')

# Cognitive. MMSE lives in OBSERVATION post domain routing (2026-08-21).
mmse_o = (get_observations('MMSE:MMSCORE', exact=True)
          .rename(columns={'observation_date': 'measurement_date'}))
features['mmse'] = (get_baseline(mmse_o).set_index('person_id')['value_as_number']
                    .reindex(features.index))
print(f"  mmse: {features['mmse'].notna().sum()} subjects (from OBSERVATION)")
features = add_baseline_measurement(features, 'CDR:CDSOB', 'cdr_sb')

# Biomarkers
features = add_baseline_measurement(features, 'PTAU217', 'ptau217')
features = add_baseline_measurement(features, 'AMYLOID\\|Florbetapir\\|Composite_Summary', 'amyloid_suvr')

# Roche neurofilament light
features = add_baseline_measurement(features, 'ROCHE:NF-L', 'nfl')

# Roche GFAP
features = add_baseline_measurement(features, 'ROCHE:GFAP', 'gfap')

# Imaging - hippocampal volume (average of left + right)
mri_lhipp = get_measurements('MRI:LeftHippocampus', exact=True)
mri_rhipp = get_measurements('MRI:RightHippocampus', exact=True)
if len(mri_lhipp) > 0 and len(mri_rhipp) > 0:
    lh_bl = get_baseline(mri_lhipp).set_index('person_id')['value_as_number'].rename('left_hipp')
    rh_bl = get_baseline(mri_rhipp).set_index('person_id')['value_as_number'].rename('right_hipp')
    hipp = pd.DataFrame({'left_hipp': lh_bl, 'right_hipp': rh_bl}).dropna()
    hipp['hippocampal_vol'] = (hipp['left_hipp'] + hipp['right_hipp']) / 2
    features['hippocampal_vol'] = hipp['hippocampal_vol'].reindex(features.index)
    print(f'  hippocampal_vol: {features["hippocampal_vol"].notna().sum()} subjects')
else:
    print(f'  hippocampal_vol: MRI data not found')

# Clinical - weight
features = add_baseline_measurement(features, 'STDWT', 'weight', exact=True)

# Clinical - systolic blood pressure
features = add_baseline_measurement(features, 'VSBPSYS', 'systolic_bp', exact=True)

In [ ]:
# --- OBSERVATION features ---
print('OBSERVATION features (baseline):')

def add_baseline_observation(features_df, source_pattern, feature_name, exact=False):
    """Extract baseline observation and add as feature column."""
    o = get_observations(source_pattern, exact=exact)
    if len(o) == 0:
        print(f'  {feature_name}: 0 records found (pattern: {source_pattern})')
        return features_df
    o = o.copy()
    o['observation_date'] = pd.to_datetime(o['observation_date'])
    idx = o.groupby('person_id')['observation_date'].idxmin()
    o_bl = o.loc[idx]
    if o_bl['value_as_number'].notna().any():
        o_map = o_bl.set_index('person_id')['value_as_number']
    else:
        o_map = o_bl.set_index('person_id')['value_as_concept_id']
    features_df[feature_name] = o_map.reindex(features_df.index)
    n_avail = features_df[feature_name].notna().sum()
    print(f'  {feature_name}: {n_avail} subjects')
    return features_df

# Lifestyle (correct source patterns from observation.csv)
features = add_baseline_observation(features, 'HABITS:SMOKE', 'smoking', exact=True)
features = add_baseline_observation(features, 'HABITS:ALCOHOL', 'alcohol', exact=True)
features = add_baseline_observation(features, 'HABITS:AEROBIC', 'exercise', exact=True)

# Family history - parental dementia (FAMHX:MOTHER or FAMHX:FATHER)
features = add_baseline_observation(features, 'FAMHX:MOTHER', 'family_hx_mother')

# APOE carrier status (from measurement: ADQS:APOEGNPRSNFLG)
features = add_baseline_measurement(features, 'ADQS:APOEGNPRSNFLG', 'apoe_e4_carrier')

# --- DRUG_EXPOSURE features ---
print('\nDRUG_EXPOSURE features:')
features['tx_solanezumab'] = tx_lookup.reindex(features.index).map(
    lambda x: 1 if x == 'Solanezumab' else (0 if x == 'Placebo' else np.nan)
)
print(f'  tx_solanezumab: {features["tx_solanezumab"].notna().sum()} subjects')

print(f'\nTotal features: {len([c for c in features.columns if c != "declined"])}')
print(f'Total subjects: {len(features)}')

In [ ]:
# ── Feature availability summary ──
feature_cols = [c for c in features.columns if c != 'declined']
availability = features[feature_cols].notna().sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(len(availability)), availability.values, color='steelblue')
ax.set_yticks(range(len(availability)))
ax.set_yticklabels(availability.index)
ax.set_xlabel('Number of Subjects with Data')
ax.set_title('Feature Availability Across OMOP Tables')
ax.axvline(x=len(features), color='red', linestyle='--', alpha=0.5, label=f'Total subjects ({len(features)})')
ax.legend()
ax.invert_yaxis()

# Color-code by OMOP table
table_colors = {
    'age': '#1f77b4', 'female': '#1f77b4', 'white': '#1f77b4',  # PERSON
    'mmse': '#ff7f0e', 'cdr_sb': '#ff7f0e',  # MEASUREMENT (cognitive)
    'ptau217': '#2ca02c', 'nfl': '#2ca02c', 'gfap': '#2ca02c',  # MEASUREMENT (biomarker)
    'amyloid_suvr': '#d62728', 'hippocampal_vol': '#d62728',  # MEASUREMENT (imaging)
    'weight': '#9467bd', 'systolic_bp': '#9467bd',  # MEASUREMENT (clinical)
    'smoking': '#8c564b', 'alcohol': '#8c564b', 'exercise': '#8c564b',  # OBSERVATION
    'family_hx_mother': '#8c564b', 'apoe_e4_carrier': '#8c564b',  # OBSERVATION
    'tx_solanezumab': '#e377c2',  # DRUG_EXPOSURE
}

for i, feat in enumerate(availability.index):
    if feat in table_colors:
        bars[i].set_color(table_colors[feat])

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis5_feature_availability.png', bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, make_scorer

# ── Prepare feature matrix ──
# Drop features with <30% availability
min_availability = 0.30
valid_features = [c for c in feature_cols if features[c].notna().mean() >= min_availability]
print(f'Features with ≥{min_availability*100:.0f}% availability: {len(valid_features)}')
print(f'Dropped: {set(feature_cols) - set(valid_features)}')

X = features[valid_features].copy()
y = features['declined'].copy()

# Drop rows where outcome is missing
mask = y.notna()
X = X[mask]
y = y[mask].astype(int)

print(f'\nFinal dataset: {len(X)} subjects, {len(valid_features)} features')
print(f'Outcome: {y.sum()} declined ({100*y.mean():.1f}%), {len(y)-y.sum()} stable')

In [ ]:
# ── Multi-modal models (all OMOP domains) ──
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42))
    ]),
    'Gradient Boosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                           learning_rate=0.1, random_state=42))
    ]),
}

print('=== Multi-Modal Prediction (All OMOP Domains) ===')
print(f'Features: {valid_features}')
print(f'Dataset: {len(X)} subjects, {y.sum()} declined ({100*y.mean():.1f}%)\n')

results_all = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    results_all[name] = scores
    print(f'{name:25s}: AUROC = {scores.mean():.3f} (±{scores.std():.3f})')

In [ ]:
# ── Single-domain comparisons ──
domain_features = {
    'Demographics only': [f for f in ['age', 'female', 'white'] if f in valid_features],
    'Cognitive only': [f for f in ['mmse', 'cdr_sb'] if f in valid_features],
    'Biomarkers only': [f for f in ['ptau217', 'nfl', 'gfap'] if f in valid_features],
    'Imaging only': [f for f in ['amyloid_suvr', 'hippocampal_vol'] if f in valid_features],
    'Lifestyle only': [f for f in ['smoking', 'alcohol', 'exercise', 'family_hx_mother', 'apoe_e4_carrier'] if f in valid_features],
}

# Use best model from multi-modal
best_model_name = max(results_all, key=lambda k: results_all[k].mean())
best_model_template = models[best_model_name]

print(f'=== Single-Domain Comparison (using {best_model_name}) ===\n')

import copy
domain_results = {}
for domain_name, domain_feats in domain_features.items():
    if len(domain_feats) == 0:
        print(f'{domain_name:25s}: No features available')
        continue
    
    X_domain = X[domain_feats]
    # Need sufficient non-null data
    mask_domain = X_domain.notna().any(axis=1)
    if mask_domain.sum() < 50:
        print(f'{domain_name:25s}: Insufficient data ({mask_domain.sum()} subjects)')
        continue
    
    model_copy = copy.deepcopy(best_model_template)
    scores = cross_val_score(model_copy, X_domain, y, cv=cv, scoring='roc_auc')
    domain_results[domain_name] = scores
    print(f'{domain_name:25s}: AUROC = {scores.mean():.3f} (±{scores.std():.3f}) [{len(domain_feats)} features]')

# Add multi-modal result
domain_results[f'Multi-modal ({best_model_name})'] = results_all[best_model_name]
print(f'\n{"Multi-modal":25s}: AUROC = {results_all[best_model_name].mean():.3f} (±{results_all[best_model_name].std():.3f}) [{len(valid_features)} features]')

In [ ]:
# ── Comparison bar plot ──
fig, ax = plt.subplots(figsize=(10, 6))

domain_names = list(domain_results.keys())
domain_means = [domain_results[d].mean() for d in domain_names]
domain_stds = [domain_results[d].std() for d in domain_names]

# Color multi-modal differently
colors_bar = ['#95a5a6'] * (len(domain_names) - 1) + ['#2ecc71']

bars = ax.barh(range(len(domain_names)), domain_means, xerr=domain_stds,
               color=colors_bar, capsize=5, edgecolor='white')
ax.set_yticks(range(len(domain_names)))
ax.set_yticklabels(domain_names)
ax.set_xlabel('AUROC (5-fold CV)')
ax.set_title('Single-Domain vs Multi-Modal Prediction of Cognitive Decline')
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.3, label='Chance')
ax.set_xlim(0.4, 1.0)
ax.legend()

# Annotate values
for i, (m, s) in enumerate(zip(domain_means, domain_stds)):
    ax.text(m + s + 0.01, i, f'{m:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis5_domain_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Feature importance (from best model) ──
import copy

final_model = copy.deepcopy(best_model_template)
final_model.fit(X, y)

# Extract importances
clf = final_model.named_steps['clf']
if hasattr(clf, 'feature_importances_'):
    importances = clf.feature_importances_
elif hasattr(clf, 'coef_'):
    importances = np.abs(clf.coef_[0])
else:
    importances = np.ones(len(valid_features))

feat_imp = pd.Series(importances, index=valid_features).sort_values(ascending=True)

# Map features to OMOP tables
table_map = {
    'age': 'PERSON', 'female': 'PERSON', 'white': 'PERSON',
    'mmse': 'MEASUREMENT\n(cognitive)', 'cdr_sb': 'MEASUREMENT\n(cognitive)',
    'ptau217': 'MEASUREMENT\n(biomarker)', 'nfl': 'MEASUREMENT\n(biomarker)', 'gfap': 'MEASUREMENT\n(biomarker)',
    'amyloid_suvr': 'MEASUREMENT\n(imaging)', 'hippocampal_vol': 'MEASUREMENT\n(imaging)',
    'weight': 'MEASUREMENT\n(clinical)', 'systolic_bp': 'MEASUREMENT\n(clinical)',
    'smoking': 'OBSERVATION', 'alcohol': 'OBSERVATION', 'exercise': 'OBSERVATION',
    'family_hx_mother': 'OBSERVATION', 'apoe_e4_carrier': 'OBSERVATION',
    'tx_solanezumab': 'DRUG_EXPOSURE',
}
table_color_map = {
    'PERSON': '#1f77b4',
    'MEASUREMENT\n(cognitive)': '#ff7f0e',
    'MEASUREMENT\n(biomarker)': '#2ca02c',
    'MEASUREMENT\n(imaging)': '#d62728',
    'MEASUREMENT\n(clinical)': '#9467bd',
    'OBSERVATION': '#8c564b',
    'DRUG_EXPOSURE': '#e377c2',
}

fig, ax = plt.subplots(figsize=(10, 8))
colors_imp = [table_color_map.get(table_map.get(f, ''), '#333333') for f in feat_imp.index]
ax.barh(range(len(feat_imp)), feat_imp.values, color=colors_imp)
ax.set_yticks(range(len(feat_imp)))
ax.set_yticklabels(feat_imp.index)
ax.set_xlabel('Feature Importance')
ax.set_title(f'Feature Importance by OMOP Domain ({best_model_name})')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in table_color_map.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8, title='OMOP Table')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'analysis5_feature_importance.png', bbox_inches='tight')
plt.show()

# Domain contribution summary
feat_imp_df = pd.DataFrame({'feature': feat_imp.index, 'importance': feat_imp.values})
feat_imp_df['domain'] = feat_imp_df['feature'].map(table_map)
domain_contrib = feat_imp_df.groupby('domain')['importance'].sum().sort_values(ascending=False)
print('\n=== Domain Contribution to Prediction ===')
for domain, contrib in domain_contrib.items():
    pct = 100 * contrib / domain_contrib.sum()
    print(f'  {domain.replace(chr(10), " "):30s}: {pct:.1f}%')

---
## Summary of Results

In [ ]:
# ── Replication scorecard ──
pub_rates = {'Low': 19, 'Intermediate': 35, 'High': 41}
auc20 = roc_auc_score((roc_df['centiloids'] >= 20).astype(int), roc_df['ptau217'])
auc33 = roc_auc_score((roc_df['centiloids'] >= 33).astype(int), roc_df['ptau217'])
best_single = max(v.mean() for k, v in domain_results.items() if 'Multi' not in k)

rows = [
    ('1 Randomized N (sol/pla)', f'{len(sol_persons)} / {len(pla_persons)}',
     '578 / 591', 'Sperling; Table 1 stats are mITT 564/583'),
    ('2 P-tau217 AUROC ≥20 / ≥33 CL', f'{auc20:.2f} / {auc33:.2f}',
     '0.87 / 0.89', 'Rissman'),
    ('2 P-tau217 vs CL, Spearman r', f'{r_spearman:.2f}', '0.73', 'Rissman'),
    ('2 CSF Aβ42/40 vs CL, Spearman r', f'{r_csf:.2f}', '−0.54', 'Rissman Fig. 1B, CSF substudy'),
    ('3 CDR progression wk240 (KM|GEE)',
     '  '.join(f'{km_results[t]["KM_rate_%"]:.0f}|{gee_pred_rates[t]:.0f}%' for t in pub_rates),
     '19 / 35 / 41%', 'Rentz; published rates are modeled'),
    ('4 PACC diff wk240',
     f'{tx_effect_full:.2f} ({ci_low_mmrm:.2f}, {ci_high_mmrm:.2f})',
     '−0.30 (−0.82, 0.22)', 'Sperling; approximated cLDA'),
    ('6 Amyloid ΔCL sol / pla', f'{amy_sol.mean():+.1f} / {amy_pla.mean():+.1f}',
     '+11.6 / +19.3', 'Sperling'),
    ('7 CDR-G progression by arm (KM)',
     f'{cdr_arm_km["Solanezumab"]:.0f}% / {cdr_arm_km["Placebo"]:.0f}%',
     '35% / 32%', 'Sperling'),
]
for name, res in secondary_results.items():
    rows.append((f'7 {name}', f'{res["diff"]:+.2f}', f'{res["published_diff"]:+.2f}',
                 'Sperling; crude t-test vs adjusted model'))
rows += [
    ('8 Subgroups: all CIs cover 0',
     str(all((r['CI_low'] <= 0 <= r['CI_high']) for _, r in sg_df.iterrows())),
     '"similar results" (Fig. S2)', 'Sperling'),
    ('11 Memory box prog. by tertile',
     ' / '.join(f'{box_progression["Memory"][t]["rate"]:.0f}%' for t in pub_rates),
     '18 / 34 / 41%', 'Rentz'),
    ('5 Multi-modal AUROC (novel)', f'{results_all[best_model_name].mean():.2f}',
     '—', f'best single domain {best_single:.2f}'),
]

scorecard = pd.DataFrame(rows, columns=['Analysis', 'OMOP', 'Published', 'Notes'])
pd.set_option('display.max_colwidth', 60)
display(scorecard)

print('\nAnalyses 9 (non-p-tau biomarker AUROCs) and 12 (pre-screening scenario) are')
print('descriptive extensions with no published comparator in the three source papers.')